# 

In [2]:
# ===================== 0) Global Config — single import cell (sklearn-first) =====================
# —— Std lib ——
import os
import sys
import re # <<<--- Added here
import json
import math
import random
import warnings
from pathlib import Path # <<<--- Added here
from datetime import datetime

# —— Data / Sci ——
import numpy as np # <<<--- Added here
import pandas as pd # <<<--- Added here

# —— Viz (optional) ——
import matplotlib.pyplot as plt

# —— ML (preprocess / models / CV) ——
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# —— statsmodels (optional) ——
try:
    import statsmodels.api as sm
    SM_AVAILABLE = True
except Exception:
    SM_AVAILABLE = False

# ===================== Reproducibility / Display =====================
SEED = 111
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 120

# ===================== Helper Functions =====================
def now_str(fmt="%Y-%m-%d %H:%M:%S"):
    return datetime.now().strftime(fmt)

def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

def print_metrics(tag, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rms_e = rmse(y_true, y_pred)
    print(f"[{tag}] MAE={mae:.2f} | RMSE={rms_e:.2f}")

def kfold_cv_scores(estimator, X, y, n_splits=6, scoring="neg_mean_absolute_error", seed=SEED):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    if scoring not in ["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"]:
        print(f"Warning: Scoring set to {scoring}, using neg_mean_absolute_error instead for CV.")
        scoring = "neg_mean_absolute_error"
    cv_score = -cross_val_score(estimator, X, y, cv=kf, scoring=scoring, n_jobs=-1).mean()
    return cv_score

def log(msg):
    print(f"[{now_str()}] {msg}")

def pretty_metrics_table(rows):
    df = pd.DataFrame(rows)
    cols = ["Model", "In-sample MAE","In-sample RMSE", "Out-of-sample MAE","Out-of-sample RMSE", "Cross-validation MAE","Cross-validation RMSE", "Kaggle Score"]
    for c in cols:
        if c not in df.columns: df[c] = ""
    if "Kaggle Score" in df.columns: cols.remove("Kaggle Score"); cols.append("Kaggle Score")
    num_cols = ["In-sample MAE", "In-sample RMSE", "Out-of-sample MAE", "Out-of-sample RMSE", "Cross-validation MAE", "Cross-validation RMSE"]
    for c in num_cols:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
             df[c] = df[c].map('{:,.2f}'.format)
    return df[cols]

# ===================== Project Paths =====================
PROJ      = Path.cwd()
DATA_RAW  = PROJ / "data" / "raw"
DATA_PROC = PROJ / "data" / "processed"
REPORTS   = PROJ / "reports"
FIGS      = REPORTS / "figures"
for p in [DATA_RAW, DATA_PROC, REPORTS, FIGS]:
    p.mkdir(parents=True, exist_ok=True)

# ===================== Assignment Constants =====================
# —— column names ——
# <<<--- NOTE: TARGET will be set dynamically based on dataset --- >>>
# TARGET = "Price" # Or "Rent" - will be set later
ID_COL = "ID"

# —— data files ——
# (These can be overridden by the file locator cell if needed)
TRAIN_FILE_PRICE = DATA_RAW / "ruc_Class25Q2_train_price.csv"
TEST_FILE_PRICE = DATA_RAW / "ruc_Class25Q2_test_price.csv"
TRAIN_FILE_RENT = DATA_RAW / "ruc_Class25Q2_train_rent.csv" # <<<--- !!! PLEASE PROVIDE ACTUAL RENT FILENAME !!!
TEST_FILE_RENT = DATA_RAW / "ruc_Class25Q2_test_price.csv"   # <<<--- !!! PLEASE PROVIDE ACTUAL RENT FILENAME !!!

KAGGLE_TEMPLATE_PATH = DATA_RAW / "submission_template_Class25Q2.csv"

# —— split / CV ——
TRAIN_SIZE = 0.80
CV_FOLDS   = 6

# —— leakage guard ——
LEAKY_KEYS = ["均价","avg","target","label","成交","挂牌","price", "rent"] # Added "rent"

# —— feature toggles ——
USE_LOG1P_NUM = False
USE_POLY      = False
POLY_DEGREE   = 2

# —— output paths ——
# (Simplified - specific names will be generated in model cells)
# SUBMISSION_PATH_FINAL = REPORTS / "submission_FINAL_BestOfBoth_MERGED.csv"

# —— OLS summary ——
RUN_SM_OLS_SUMMARY = True and SM_AVAILABLE

log(f"ENV READY | seed={SEED} | py={sys.version.split()[0]} | sklearn OK | statsmodels={SM_AVAILABLE}")
log(f"Project Root: {PROJ}")
# Target will be set during data loading

[2025-10-30 00:45:54] ENV READY | seed=111 | py=3.12.4 | sklearn OK | statsmodels=True
[2025-10-30 00:45:54] Project Root: /Users/pxy/Desktop/python midterm


In [5]:
# ===================================================================
# 1. Load RENT Data & Initial Cleanup
# ===================================================================

# --- Define Target for this specific dataset ---
TARGET = "Price" 
log(f"Setting TARGET for this run: {TARGET}")

# --- Helper functions for this cell ---
def normalize_cols(cols):
    """Strips whitespace and removes internal spaces from column names."""
    out = []
    for c in cols:
        c = str(c).replace("\u3000", " ").strip()
        c = re.sub(r"\s+", "", c) # Remove internal spaces too
        out.append(c)
    return out

def find_csv(pattern_keywords, candidates):
    """Finds CSV files matching keywords in candidate directories."""
    kws = [k.lower() for k in pattern_keywords]
    hits = []
    log(f"Searching for CSVs matching: {kws}")
    for root in candidates:
        if not root.is_dir(): # Check if it's a directory
             log(f"  Skipping non-existent or non-directory: {root}")
             continue
        log(f"  Scanning in: {root}")
        # Use rglob for recursive search
        for p in root.rglob("*.csv"):
            name = p.name.lower()
            if all(k in name for k in kws):
                hits.append(p)
                log(f"    Found candidate: {p.relative_to(PROJ)}")
    # Sort by modification time, newest first
    hits = sorted(hits, key=lambda p: p.stat().st_mtime, reverse=True)
    return hits

# --- Locate Rent Files ---
log("Locating RENT train/test CSV files...")
search_roots = [
    DATA_RAW, # Defined in Cell 0
    PROJ / "data", # One level up from raw
    PROJ,      # Project root
    # Add other potential locations if needed, using relative paths preferably
]

# Use keywords expected in the RENT filenames
train_hits = find_csv(["train", "price"], search_roots)
test_hits  = find_csv(["test", "price"],  search_roots)

print("\n--- Potential RENT Train files (newest first) ---")
for i,p in enumerate(train_hits[:3]): print(f"  {i+1}. {p.relative_to(PROJ)}")
print("--- Potential RENT Test files (newest first) ---")
for i,p in enumerate(test_hits[:3]): print(f"  {i+1}. {p.relative_to(PROJ)}")

# --- Select and Load ---
if not train_hits:
    log(f"ERROR: No RENT training CSV found matching ['train', 'rent']. Trying default path.")
    # Fallback to default from Cell 0 if locator fails
    train_fp = TRAIN_FILE_RENT # <<<--- Make sure this is correctly set in Cell 0
    if not train_fp.exists(): raise FileNotFoundError("Cannot find RENT training file.")
else:
    train_fp = train_hits[0] # Use the newest found file

if not test_hits:
    log(f"ERROR: No RENT test CSV found matching ['test', 'rent']. Trying default path.")
    test_fp = TEST_FILE_RENT # <<<--- Make sure this is correctly set in Cell 0
    if not test_fp.exists(): raise FileNotFoundError("Cannot find RENT test file.")
else:
    test_fp = test_hits[0]

log(f"Using RENT train file: {train_fp.resolve()}")
log(f"Using RENT test file : {test_fp.resolve()}")

# --- Read CSVs and Normalize Columns ---
try:
    df_raw = pd.read_csv(train_fp, low_memory=False)
    df_raw.columns = normalize_cols(df_raw.columns)

    df_test_raw = pd.read_csv(test_fp, low_memory=False)
    df_test_raw.columns = normalize_cols(df_test_raw.columns)
except Exception as e:
    log(f"ERROR reading CSV files: {e}")
    raise

log(f"RENT Train data loaded: {df_raw.shape}")
log(f"RENT Test data loaded : {df_test_raw.shape}")

# --- Check TARGET and ID ---
if TARGET not in df_raw.columns:
    log(f"ERROR: RENT Training data is missing the target column '{TARGET}'")
    log(f"Available columns: {df_raw.columns.tolist()}")
    raise KeyError(f"Target column '{TARGET}' not found in training data.")

if ID_COL not in df_test_raw.columns:
    # Generate synthetic IDs starting from a large number to avoid clashes
    start_id = 2_000_000
    df_test_raw[ID_COL] = np.arange(start_id, start_id + len(df_test_raw))
    log(f"'{ID_COL}' column not found in RENT test data -> Created synthetic IDs starting from {start_id}.")

# --- Preview Data ---
print("\n--- RENT Train Data Preview ---")
display(df_raw.head(3))
print(f"\n--- RENT Target ({TARGET}) Distribution ---")
display(df_raw[TARGET].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).apply('{:,.2f}'.format))

[2025-10-30 00:45:56] Setting TARGET for this run: Price
[2025-10-30 00:45:56] Locating RENT train/test CSV files...
[2025-10-30 00:45:56] Searching for CSVs matching: ['train', 'price']
[2025-10-30 00:45:56]   Scanning in: /Users/pxy/Desktop/python midterm/data/raw
[2025-10-30 00:45:56]   Scanning in: /Users/pxy/Desktop/python midterm/data
[2025-10-30 00:45:56]   Scanning in: /Users/pxy/Desktop/python midterm
[2025-10-30 00:45:56]     Found candidate: ruc_Class25Q2_train_price.csv
[2025-10-30 00:45:56]     Found candidate: .ipynb_checkpoints/ruc_Class25Q2_train_price-checkpoint.csv
[2025-10-30 00:45:56] Searching for CSVs matching: ['test', 'price']
[2025-10-30 00:45:56]   Scanning in: /Users/pxy/Desktop/python midterm/data/raw
[2025-10-30 00:45:56]   Scanning in: /Users/pxy/Desktop/python midterm/data
[2025-10-30 00:45:56]   Scanning in: /Users/pxy/Desktop/python midterm
[2025-10-30 00:45:56]     Found candidate: ruc_Class25Q2_test_price.csv
[2025-10-30 00:45:56]     Found candidate:

,城市,区域,板块,环线,Price,房屋户型,所在楼层,建筑面积,套内面积,房屋朝向,建筑结构,装修情况,梯户比例,配备电梯,别墅类型,交易时间,交易权属,上次交易,房屋用途,房屋年限,产权所属,抵押信息,房屋优势,核心卖点,户型介绍,周边配套,交通出行,lon,lat,年份,区县,板块_comm,环线位置,物业类别,建筑年代,开发商,房屋总数,楼栋总数,物业公司,绿化率,容积率,物业费,建筑结构_comm,物业办公电话,产权描述,供水,供暖,供电,燃气费,供热费,停车位,停车费用,coord_x,coord_y,客户反馈
0,0,109.0,150.0,二至三环,6.194049e+06,2室1厅1厨1卫,中楼层 (共5层),52.3㎡,NaN,南 北,混合结构,精装,一梯三户,无,NaN,2021-03-29,商品房,2013-07-31,普通住宅,满五年,非共有,NaN,装修、房本满五年,此房是南北通透小板楼，户型方正，格局合理,房子是南北通透户型方正采光好，前后没有遮挡视野好，通风效果好,医院、公园、超市，生活便利，火箭军医院、积水潭医院，双秀公园，人定湖公园，物美超市、世纪华联等。,NaN,117.424278,40.975752,2018.0,109.0,150.0,二至三环,普通住宅/平房,1955-2000年,无开发商,1317户,19栋,北京首华物业管理有限公司,30%,3.00,1.3-1.65元/月/㎡,板楼/平房,NaN,商品房/已购公房/央产房/私产,民水,集中供暖,民电,2.61元/m³,30元/㎡,300.0,暂无,117.424278,40.975752,听说，设施老旧，停车费高
1,0,65.0,299.0,五至六环,4.354153e+06,3室1厅1厨1卫,顶层 (共6层),127.44㎡,123.7㎡,南 北,混合结构,精装,一梯两户,无,NaN,2020-10-29,商品房,2010-12-10,普通住宅,满五年,非共有,NaN,装修、房本满五年,南北通透商品房自住装修无个税,房子三居一卫，户型方正，南北通透，客厅朝南带阳台，主卧朝南，东向有窗户，次卧、厨房朝北，厨房...,医院：北京京都儿童医院、昌平中西结合医院，积水潭医院配套设施：美廉美，工商，农行，邮政等银行...,NaN,117.389228,41.091295,2017.0,65.0,299.0,五至六环,普通住宅/商业/底商,2005年,北京首都开发控股有限公司,2317户,40栋,北京天鸿宝地物业管理经营有限公司,30%,1.73,0.65元/月/㎡,板楼,010-81738522,商品房/一类经济适用房/私产,商水/民水,自采暖,商电/民电,2.61元/m³,NaN,1550.0,150,117.389228,41.091295,整体印象，网速快，面积适中
2,0,62.0,911.0,五至六环,3.321992e+06,3室2厅1厨2卫,低楼层 (共6层),118.02㎡,101.95㎡,东南,钢混结构,简装,一梯五户,有,NaN,2020-11-24,商品房,2013-10-28,普通住宅,满五年,非共有,NaN,地铁、装修、房本满五年,房子满五年，商品房，三居室两个卫生间,此房户型方正，没有浪费的面积，东南朝向，客厅，主卧朝南，两间次卧朝东，两个卫生间，干湿分离。...,NaN,NaN,117.200934,40.747919,2018.0,62.0,911.0,五至六环,普通住宅/写字楼/商业/底商/地下仓储/库房,2011-2012年,中昂地产(集团)有限公司,1554户,20栋,北京中昂物业管理有限公司,30%,1.70,1.98-2.98元/月/㎡,板楼,010-69365066,商品房/限价商品房,商水/民水,集中供暖/自采暖,商电/民电,2.61元/m³,30元/㎡,324.0,150,117.200934,40.747919,地段一般，停车划线清晰，说白了，居住体验佳



--- RENT Target (Price) Distribution ---


count       103,871.00
mean      2,262,366.07
std       2,532,925.40
min          74,553.30
1%          254,478.05
5%          429,323.13
25%         891,091.30
50%       1,479,407.11
75%       2,680,757.04
95%       6,499,318.04
99%      12,990,538.99
max      56,226,431.30
Name: Price, dtype: object

In [6]:
# ===================== 1) Load Data & Normalize Columns (fixed) =====================
from pathlib import Path
import re, numpy as np, pandas as pd

train_fp = Path("/Users/pxy/Desktop/python midterm/ruc_Class25Q2_train_rent.csv")
test_fp  = Path("/Users/pxy/Desktop/python midterm/ruc_Class25Q2_test_rent.csv")

def normalize_cols(cols):
    """去首尾空格、全角空格→半角、删除所有空格（如 '绿 化 率'→'绿化率'）"""
    out = []
    for c in cols:
        c = str(c).replace("\u3000", " ").strip()
        c = re.sub(r"\s+", "", c)
        out.append(c)
    return out

# read
df = pd.read_csv(train_fp)
df.columns = normalize_cols(df.columns)  
log(f"train loaded: {train_fp.name}, shape={df.shape}")

df_test = pd.read_csv(test_fp)
df_test.columns = normalize_cols(df_test.columns)
log(f"test loaded:  {test_fp.name}, shape={df_test.shape}")

# ID filling
assert TARGET in df.columns, f"未找到目标列：{TARGET}"
if ID_COL not in df.columns:
    df[ID_COL] = np.arange(len(df))
    log(f"{ID_COL} not found in train, generated sequential IDs.")
if ID_COL not in df_test.columns:
    df_test[ID_COL] = np.arange(10_000_000, 10_000_000 + len(df_test))
    log(f"{ID_COL} not found in test, generated synthetic IDs.")

print("\n=== Column name check (train ∩ test) ===")
common = set(df.columns) & set(df_test.columns)
print(f"common cols: {len(common)} | train only: {len(df.columns)-len(common)} | test only: {len(df_test.columns)-len(common)}")

print("\n=== TARGET quick stats (train) ===")
print(df[TARGET].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]))

print("\n=== Head (train) ===")
display(df.head(3))

print("\n=== Head (test) ===")
display(df_test.head(3))


[2025-10-30 00:45:59] train loaded: ruc_Class25Q2_train_rent.csv, shape=(98899, 46)
[2025-10-30 00:45:59] test loaded:  ruc_Class25Q2_test_rent.csv, shape=(9773, 46)
[2025-10-30 00:45:59] ID not found in train, generated sequential IDs.

=== Column name check (train ∩ test) ===
common cols: 46 | train only: 1 | test only: 0

=== TARGET quick stats (train) ===
count    9.889900e+04
mean     5.829090e+05
std      6.218424e+05
min      1.793807e+04
1%       9.297267e+04
5%       1.383090e+05
50%      3.949369e+05
95%      1.533062e+06
99%      3.227491e+06
max      1.540419e+07
Name: Price, dtype: float64

=== Head (train) ===


,城市,户型,装修,Price,楼层,面积,朝向,交易时间,付款方式,租赁方式,电梯,车位,用水,用电,燃气,采暖,租期,配套设施,lon,lat,年份,区县,板块,环线位置,物业类别,建筑年代,开发商,房屋总数,楼栋总数,物业公司,绿化率,容积率,物业费,建筑结构,物业办公电话,产权描述,供水,供暖,供电,燃气费,供热费,停车位,停车费用,coord_x,coord_y,客户反馈,ID
0,0,1室1厅1卫,精装修,654646.481811,4/6层,36.42㎡,西,2024-11-28,季付价,整租,无,NaN,民水,民电,有,集中供暖,1年,洗衣机、空调、衣柜、热水器、床、宽带,117.336830,40.930871,2022.0,81.0,145.0,三至四环,普通住宅/底商,1963-2001年,中国房地产开发北京有限公司,1731户,19栋,北京中房物业管理有限公司,30%,2.5,1.1-1.85元/月/㎡,塔楼/板楼,010-68185279,商品房/已购公房/央产房/二类经济适用房/私产,民水,集中供暖,民电,2.61元/m³,24-30元/㎡,450.0,150,117.339283,40.930007,潮气重，仔细一看，房屋保养好,0
1,0,1室1厅1卫,精装修,665412.057415,4/6层,41.00㎡,南,2024-10-30,季付价,整租,无,NaN,民水,民电,有,集中供暖,NaN,洗衣机、空调、衣柜、电视、热水器、床、宽带,117.450170,40.876186,2022.0,7.0,581.0,二至三环,普通住宅/商业/底商,1988-2002年,北京城建集团,1931户,9栋,北京美加兴业物业管理有限公司,30%,1.2,0.6-1.15元/月/㎡,塔楼/板楼/塔板结合,NaN,商品房/已购公房/使用权/私产,民水,集中供暖,民电,2.61元/m³,30元/㎡,150.0,150,117.446526,40.876743,服务响应中等，看起来，管线老化，消防设施齐全,1
2,0,1室1厅1卫,精装修,778222.820548,1/18层,37.36㎡,北,2024-11-12,季付价,整租,有,租用车位,民水,民电,有,集中供暖,NaN,洗衣机、空调、衣柜、电视、热水器、床、宽带,117.516502,40.903514,2022.0,68.0,355.0,三至四环,车库/普通住宅/写字楼/商业/底商,2004-2009年,北京中力房地产开发有限公司,1891户,12栋,北京颐中国际物业管理有限公司,30%,2.7,2-2.66元/月/㎡,塔楼/板楼/塔板结合,010-58610780,商品房/私产,商水/民水,集中供暖,商电/民电,2.61-2.63元/m³,30-46元/㎡,965.0,500,117.518524,40.905357,差不多这样，电梯新，总的来说，宽敞，性价比高,2



=== Head (test) ===


,ID,城市,户型,装修,楼层,面积,朝向,交易时间,付款方式,租赁方式,电梯,车位,用水,用电,燃气,采暖,租期,配套设施,lon,lat,年份,区县,板块,环线位置,物业类别,建筑年代,开发商,房屋总数,楼栋总数,物业公司,绿化率,容积率,物业费,建筑结构,物业办公电话,产权描述,供水,供暖,供电,燃气费,供热费,停车位,停车费用,coord_x,coord_y,客户反馈
0,2000000,1,2室2厅1卫,精装修,低楼层/18层,86.94㎡,南 北,2025-08-01,NaN,整租,有,NaN,民水,民电,有,自采暖,NaN,洗衣机、空调、衣柜、电视、冰箱、热水器、床、暖气、宽带、天然,117.345687,40.447235,2023.0,24.0,303.0,NaN,普通住宅,2011-2019年,固安县兴源房地产开发有限公司,992户,12栋,固安县兴业源物业服务有限公司,40%,2.5,1.5-1.99元/月/㎡,塔板结合,NaN,商品房,民水,自采暖,民电,2.15元/m³,NaN,1600.0,100,117.345687,40.447235,电费按表计量，治安堪忧，拎包入住
1,2000001,10,2室1厅1卫,精装修,低楼层/8层,72.60㎡,南,2025-05-23,月付价,整租,有,NaN,民水,民电,有,NaN,NaN,洗衣机、空调、衣柜、冰箱、热水器、床,114.279469,24.158183,2022.0,105.0,985.0,NaN,普通住宅,1985-2008年,无开发商,806户,15栋,NaN,25%,3.0,NaN,塔楼/板楼,无,商品房/房改房,民水,NaN,民电,3.45元/m³,NaN,200.0,800,114.279469,24.158183,简单来说，交通方便，体感上还好，邻居和善
2,2000002,3,2室2厅1卫,精装修,中楼层/20层,98.00㎡,南,2025-02-18,NaN,整租,有,租用车位,NaN,NaN,有,NaN,1年以内,洗衣机、空调、衣柜、电视、冰箱、热水器、床、天然气,121.684578,32.198660,2022.0,21.0,387.0,NaN,NaN,NaN,NaN,1327户,13栋,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.684578,32.198660,公用设施正常，绿化分布均匀


In [8]:
# ===================================================================
# A. (已修复) 清洗 & 特征工程工具箱
# ===================================================================
import re, numpy as np, pandas as pd
from IPython.display import display
import time

# --- 确保 log 函数已定义 ---
def log(s):
    """一个简单的带时间戳的打印函数"""
    print(f"[{time.strftime('%H:%M:%S')}] {s}")

def _to_num(s):
    if pd.isna(s): return np.nan
    s = str(s)
    m = re.findall(r"(-?\d+(?:\.\d+)?)\s*[-~—]\s*(-?\d+(?:\.\d+)?)", s)
    if m:
        a, b = map(float, m[0]);  
        return (a + b) / 2.0
    m = re.findall(r"-?\d+(?:\.\d+)?", s)
    return float(m[0]) if m else np.nan

def _pct_to_float(s):
    if pd.isna(s): return np.nan
    s = str(s)
    if "%" in s:
        v = _to_num(s)
        return v/100 if pd.notna(v) else np.nan
    return _to_num(s)

# <--- 【修改 1】语法错误：在这里添加了换行
def _parse_layout(s):
    d = dict(卧室数=0, 客厅数=0, 卫生间数=0, 厨房数=1)
    if not isinstance(s, str): return pd.Series(d)
    m = re.findall(r'(\d+)\s*室', s);  d['卧室数']   = int(m[0]) if m else d['卧室数']
    m = re.findall(r'(\d+)\s*厅', s);  d['客厅数']   = int(m[0]) if m else d['客厅数']
    m = re.findall(r'(\d+)\s*卫', s);  d['卫生间数'] = int(m[0]) if m else d['卫生间数']
    m = re.findall(r'(\d+)\s*厨', s);  d['厨房数']   = int(m[0]) if m else d['厨房数']
    if re.search(r'0\s*厨', s): d['厨房数'] = 0
    return pd.Series(d)

def _parse_floor(s):
    res = dict(当前楼层=np.nan, 总楼层=np.nan, 楼层级别=0, 是否顶层=0, 是否底层=0)
    if not isinstance(s, str): return pd.Series(res)
    m = re.findall(r'(\d+)\s*/\s*(\d+)\s*层', s)
    if m:
        cur, tot = map(int, m[0])
        res.update(当前楼层=cur, 总楼层=tot, 是否顶层=int(cur==tot), 是否底层=int(cur==1))
    lvl_map = {'低':1, '中':2, '高':3}
    for k,v in lvl_map.items():
        if k in s: res["楼层级别"] = v
    m2 = re.findall(r'共(\d+)层', s)
    if m2 and pd.isna(res["总楼层"]): res["总楼层"] = int(m2[0])
    return pd.Series(res)

def _add_orientation_flags(df, col="朝向"):
    if col not in df.columns: return df
    for k in ['东','南','西','北','东南','东北','西南','西北']:
        df[f'朝{k}'] = df[col].astype(str).str.contains(k).astype(int)
    return df

def _parse_pay_cycle(s):
    if pd.isna(s): return np.nan, 0, 0, 0
    t = str(s)
    m = re.search(r'付[一二两三四五六七八九十\d]+', t)
    if m:
        trans = {'一':1,'二':2,'两':2,'三':3,'四':4,'五':5,'六':6,'七':7,'八':8,'九':9,'十':10}
        digits = re.findall(r'(\d+)|([一二两三四五六七八九十])', m.group(0))
        if digits:
            num = None
            for d,cn in digits:
                if d: num = int(d)
                elif cn: num = trans.get(cn, None) if num is None else num
            if num in [1,3,6,12]:
                return float(num), int(num==3), int(num==1), int(num==12)
    if "月付" in t:   return 1.0, 0, 1, 0
    if "季付" in t:   return 3.0, 1, 0, 0
    if "半年付" in t: return 6.0, 0, 0, 0
    if "年付" in t:   return 12.0, 0, 0, 1
    return np.nan, 0, 0, 0

def normalize_frame(raw):
    df = raw.copy()
    
    # 1. [已修复] 数值化 (物 业 费, 绿 化 率, 面积 等)
    for c in ["Price","面积","lon","lat","coord_x","coord_y","停车位","停车费用","物业费","燃气费","供热费","容积率", "物 业 费"]:
        if c in df.columns: df[c] = df[c].apply(_to_num)
        
    if "绿化率" in df.columns: df["绿化率"] = df["绿化率"].apply(_pct_to_float)
    if "绿 化 率" in df.columns: df["绿 化 率"] = df["绿 化 率"].apply(_pct_to_float)
    
    for c in ["房屋总数","楼栋总数","建筑年代","年份"]:
        if c in df.columns: df[c] = pd.to_numeric(df[c].apply(_to_num), errors="coerce")

    # 2. [已修复] 结构化文字 (户型, 楼层, 朝向)
    if "户型" in df.columns: df = pd.concat([df, df["户型"].apply(_parse_layout)], axis=1)
    if "楼层" in df.columns: df = pd.concat([df, df["楼层"].apply(_parse_floor)], axis=1)
    df = _add_orientation_flags(df, "朝向")

    # 3. [已修复] 时间 (挂牌时间 or 交易时间)
    time_col = "交易时间" if "交易时间" in df.columns else "挂牌时间"
    if time_col in df.columns:
        dt = pd.to_datetime(df[time_col].astype(str), errors="coerce")
        df["year"] = dt.dt.year; df["month"] = dt.dt.month; df["quarter"] = dt.dt.quarter

    # 4. [已修复] 付款方式
    if "付款方式" in df.columns:
        tmp = df["付款方式"].apply(_parse_pay_cycle).apply(pd.Series)
        tmp.columns = ["pay_cycle_months","is_quarterly","is_monthly","is_yearly"]
        df = pd.concat([df, tmp], axis=1)

    # 5. [已修复] 供暖/采暖
    def _heat_flags(row):
        txt = f"{row.get('采暖','')}{row.get('供暖','')}"
        return pd.Series({
            "central_heating": int("集中" in txt),
            "self_heating":    int(("自" in txt) or ("分散" in txt))
        })
    df = pd.concat([df, df.apply(_heat_flags, axis=1)], axis=1)

    # 6. [已修复] 电梯
    if "电梯" in df.columns:
        df["有电梯"] = df["电梯"].astype(str).str.contains("有").astype(int)
    
    # 7. [已修复] 装修 
    if "装修" in df.columns:
        df["is_精装"] = df["装修"].astype(str).str.contains("精装").astype(int)
        df["is_简装"] = df["装修"].astype(str).str.contains("简装").astype(int)
        df["is_毛坯"] = df["装修"].astype(str).str.contains("毛坯").astype(int)

    # 8. [已修复] 车位
    if "车位" in df.columns:
        s = df["车位"].astype(str)
        df["租用车位"] = s.str.contains("租用").astype(int)
        df["免费车位"] = s.str.contains("免费").astype(int)

    # 9. [已修复] 用电/供电
    def _power_flags(row):
        txt = f"{row.get('用电','')}{row.get('供电','')}{row.get('用水','')}{row.get('供水','')}"
        return pd.Series({
            "is_民水": int("民水" in txt),
            "is_商水": int("商水" in txt),
            "is_民电": int("民电" in txt),
            "is_商电": int("商电" in txt)
        })
    df = pd.concat([df, df.apply(_power_flags, axis=1)], axis=1)

    # 10. [已修复] 配套设施
    if "配套设施" in df.columns:
        s = df["配套设施"].astype(str).str.strip()
        is_empty = s.eq("") | s.str.contains(r"^无$", regex=True)
        sep_cnt = s.str.count(r"[、，,；;]")
        df["设施数"] = np.where(is_empty, 0, sep_cnt + 1)
        df["设施文本长度"] = np.where(is_empty, 0, s.str.len())
        
    # 11. [已修复] 燃气
    if "燃气" in df.columns:
        df["有燃气"] = df["燃气"].astype(str).str.contains("有").astype(int)
    
    # <--- 【修改 2】`NoneType` 错误修复：
    # 必须返回处理后的 DataFrame，否则 B 步骤会收到 None
    return df

# --- (A 步骤的剩余部分) ---
def drop_leaky_cols(df, target, id_col, keys=None):
    keys = keys or ["均价","avg","target","label","成交","挂牌","price"]
    bad = [c for c in df.columns if any(k in str(c).lower() for k in keys)]
    bad = [c for c in bad if c not in [target, id_col]]
    if bad:
        log(f"[leak-guard] drop columns: {bad}")
        df = df.drop(columns=bad)
    return df

def detect_geo_cols(df):
    def _hit(cands): return next((c for c in cands if c in df.columns), None)
    city = _hit(["城市"]); district = _hit(["区县","区域"])
    blk_cands = ["板块","商圈","片区","板块名称","商圈名称","片区名称"]
    blk_cands += [c for c in df.columns if re.search(r"(板|版)块|商圈|片区", c)]
    block = _hit(blk_cands)
    return city, district, block

def build_geo_encoders(df_tr, city, district, block, topk=30):
    enc = {}
    if city:     enc["city_vc"] = df_tr[city].value_counts()
    if district: enc["district_vc"] = df_tr[district].value_counts()
    if block:
        vc = df_tr[block].value_counts()
        enc["block_vc"] = vc
        enc["block_topk"] = list(vc.nlargest(topk).index)
    return enc

def apply_geo_encoders(df, enc, city, district, block):
    out = df.copy()
    if city and "city_vc" in enc:
        out[f"{city}_freq"] = out[city].map(enc["city_vc"]).fillna(0)
    if district and "district_vc" in enc:
        out[f"{district}_freq"] = out[district].map(enc["district_vc"]).fillna(0)
    if block and "block_vc" in enc:
        out[f"{block}_freq"] = out[block].map(enc["block_vc"]).fillna(0)
        topk = set(enc.get("block_topk", []))
        for v in topk:
            out[f'{block}=={v}'] = (out[block] == v).astype(int)
        if len(topk)>0:
            out[f'{block}==OTHER'] = (~out[block].isin(topk)).astype(int)
    return out

# <--- 【修改 3】`TypeError` 错误修复：
# 函数定义需要和 B 步骤的调用 (df, target) 相匹配
def outlier_filter_train(df, target_col):
    d = df.copy(); n0 = len(d)
    m = pd.Series(True, index=d.index)
    
    # 自动检测 target_col 是否存在
    if target_col not in d.columns:
        log(f"[outliers] 警告: 找不到目标列 '{target_col}'。跳过基于 target 的过滤。")
        target_col = None # 设为 None 以便后续检查
        
    if target_col: 
        m &= pd.to_numeric(d[target_col], errors="coerce") > 0
    if "面积" in d:  
        m &= pd.to_numeric(d["面积"], errors="coerce") > 0
        
    for c in ["年份","建筑年代"]:
        if c in d.columns:
            v = pd.to_numeric(d[c], errors="coerce")
            m &= v.between(1950, 2025) | v.isna()
            
    d = d.loc[m].copy()
    
    if target_col and "面积" in d:
        d["__ppsm__"] = pd.to_numeric(d[target_col], errors="coerce") / pd.to_numeric(d["面积"], errors="coerce")
        city, dist, _ = detect_geo_cols(d); key = city or dist
        if key:
            keep = pd.Series(True, index=d.index)
            for g, idx in d.groupby(key).groups.items():
                s = d.loc[idx, "__ppsm__"].replace([np.inf,-np.inf], np.nan).dropna()
                if len(s) < 20: continue
                q1,q3 = s.quantile([0.25,0.75]); iqr = q3-q1
                low, high = q1-3*iqr, q3+3*iqr
                keep.loc[idx] = d.loc[idx, "__ppsm__"].between(low, high) | d.loc[idx, "__ppsm__"].isna()
            d = d.loc[keep].copy()
        d.drop(columns="__ppsm__", errors="ignore", inplace=True)
        
    log(f"[outliers] kept {len(d)}/{n0}")
    return d.reset_index(drop=True)

log("工具箱 [A] (已修复) 定义完成。")

[00:45:59] 工具箱 [A] (已修复) 定义完成。


In [11]:
# ===================================================================
# B. 加载、清洗与特征工程
# ===================================================================

# 1. 定义文件常量
TRAIN_FILE = "ruc_Class25Q2_train_rent.csv"
TEST_FILE  = "ruc_Class25Q2_test_rent.csv"
TARGET = "Price"  # 你的目标列 (Y)
ID_COL = "ID"      # 你的 ID 列

# (假设 LEAKY_KEYS 和 log 函数已在 0 步骤定义)
# LEAKY_KEYS = ["均价", "avg", "target", "label", "成交", "挂牌", "price"]

# 2. 加载数据
log("加载新数据...")
try:
    df      = pd.read_csv(TRAIN_FILE, low_memory=False)
    df_test = pd.read_csv(TEST_FILE,  low_memory=False)
except FileNotFoundError:
    log(f"错误: 找不到文件 {TRAIN_FILE} 或 {TEST_FILE}。请检查路径。")
    # 如果在 notebook 中，你可能希望在这里停止
    raise

log(f"原始 train={df.shape}, test={df_test.shape}")

# 3. 规范化字段 (应用 A 步骤的 normalize_frame)
log("规范化字段 (normalize_frame)...")
train_clean = normalize_frame(df)
test_clean  = normalize_frame(df_test)

# 4. 去潜在泄漏列 (应用 A 步骤的 drop_leaky_cols)
log("移除潜在泄漏列...")
train_clean = drop_leaky_cols(train_clean, TARGET, ID_COL, LEAKY_KEYS)
test_clean  = drop_leaky_cols(test_clean,  TARGET, ID_COL, LEAKY_KEYS)

# 5. 拟合并应用地理编码器 (应用 A 步骤的函数)
log("应用地理编码...")
city_col, dist_col, block_col = detect_geo_cols(train_clean)
encoders = build_geo_encoders(train_clean, city_col, dist_col, block_col, topk=30)
train_fe  = apply_geo_encoders(train_clean, encoders, city_col, dist_col, block_col)
test_fe   = apply_geo_encoders(test_clean,  encoders, city_col, dist_col, block_col)

# 6. 只对训练集做异常值清洗 (应用 A 步骤的 outlier_filter_train)
log("过滤训练集异常值...")
# vvvv 【已修复】 vvvv
train_fe = outlier_filter_train(train_fe, TARGET) 
# ^^^^ 【已修复】 ^^^^
# 7. 分离 X (特征) 和 y (目标)
log("分离 X 和 y...")
if TARGET not in train_fe.columns:
    raise KeyError(f"目标列 '{TARGET}' 在清洗后丢失，请检查列名。")
if ID_COL not in test_fe.columns:
    log(f"警告: ID 列 '{ID_COL}' 不在测试集中，将使用索引。")
    df_test[ID_COL] = df_test.index # 创建一个备用 ID

# (!!!) 关键：这里创建了 X_all 和 y_all
y_all = train_fe[TARGET].astype(float).copy()
X_all = train_fe.drop(columns=[TARGET])
X_te  = test_fe.copy()

# 8. 检查训练/测试列一致性（关键步骤）
log("对齐 Train/Test 列...")
common_cols = sorted(set(X_all.columns) | set(X_te.columns))
X_all = X_all.reindex(columns=common_cols, fill_value=0)
X_te  = X_te.reindex(columns=common_cols,  fill_value=0)



log(f"[B] FE 完成。 X_all={X_all.shape}, X_te={X_te.shape}, y={y_all.shape}")
display(X_all.head(3))

[00:46:00] 加载新数据...
[00:46:01] 原始 train=(98899, 46), test=(9773, 46)
[00:46:01] 规范化字段 (normalize_frame)...
[00:46:50] 移除潜在泄漏列...
[00:46:50] 应用地理编码...
[00:46:50] 过滤训练集异常值...
[00:46:51] [outliers] kept 98135/98899
[00:46:51] 分离 X 和 y...
[00:46:51] 对齐 Train/Test 列...
[00:46:51] [B] FE 完成。 X_all=(98135, 119), X_te=(9773, 119), y=(98135,)


,ID,central_heating,coord_x,coord_y,is_monthly,is_quarterly,is_yearly,is_商水,is_商电,is_毛坯,is_民水,is_民电,is_简装,is_精装,lat,lon,month,pay_cycle_months,quarter,self_heating,year,交易时间,产权描述,付款方式,供暖,供水,供热费,供电,停车位,停车费用,免费车位,区县,区县_freq,卧室数,卫生间数,厨房数,城市,城市_freq,客厅数,客户反馈,容 积 率,年份,建筑年代,建筑结构,开发商,当前楼层,总楼层,户型,房屋总数,是否底层,是否顶层,有燃气,有电梯,朝东,朝东北,朝东南,朝北,朝南,朝向,朝西,朝西北,朝西南,板块,板块==1002.0,板块==1044.0,板块==1045.0,板块==1176.0,板块==1177.0,板块==138.0,板块==219.0,板块==237.0,板块==265.0,板块==303.0,板块==310.0,板块==348.0,板块==349.0,板块==387.0,板块==39.0,板块==445.0,板块==472.0,板块==484.0,板块==572.0,板块==585.0,板块==621.0,板块==628.0,板块==66.0,板块==736.0,板块==766.0,板块==769.0,板块==770.0,板块==864.0,板块==921.0,板块==987.0,板块==OTHER,板块_freq,楼层,楼层级别,楼栋总数,燃气,燃气费,物 业 费,物业公司,物业办公电话,物业类别,环线位置,用水,用电,电梯,租期,租用车位,租赁方式,绿 化 率,装修,设施数,设施文本长度,车位,配套设施,采暖,面积
0,0,1,117.339283,40.930007,0.0,1.0,0.0,0,0,0,1,1,0,1,40.930871,117.336830,11,3.0,4,0,2024,2024-11-28,商品房/已购公房/央产房/二类经济适用房/私产,季付价,集中供暖,民水,27.0,民电,450.0,150.0,0,81.0,2438.0,1,1,1,0,14542,1,潮气重，仔细一看，房屋保养好,2.5,2022.0,1982.0,塔楼/板楼,中国房地产开发北京有限公司,4.0,6.0,1室1厅1卫,1731.0,0.0,0.0,1,0,0,0,0,0,0,西,1,0,0,145.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,76.0,4/6层,0.0,19.0,有,2.61,1.475,北京中房物业管理有限公司,010-68185279,普通住宅/底商,三至四环,民水,民电,无,1年,0,整租,0.3,精装修,6,18,NaN,洗衣机、空调、衣柜、热水器、床、宽带,集中供暖,36.42
1,0,1,117.446526,40.876743,0.0,1.0,0.0,0,0,0,1,1,0,1,40.876186,117.450170,10,3.0,4,0,2024,2024-10-30,商品房/已购公房/使用权/私产,季付价,集中供暖,民水,30.0,民电,150.0,150.0,0,7.0,1539.0,1,1,1,0,14542,1,服务响应中等，看起来，管线老化，消防设施齐全,1.2,2022.0,1995.0,塔楼/板楼/塔板结合,北京城建集团,4.0,6.0,1室1厅1卫,1931.0,0.0,0.0,1,0,0,0,0,0,1,南,0,0,0,581.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,59.0,4/6层,0.0,9.0,有,2.61,0.875,北京美加兴业物业管理有限公司,NaN,普通住宅/商业/底商,二至三环,民水,民电,无,NaN,0,整租,0.3,精装修,7,21,NaN,洗衣机、空调、衣柜、电视、热水器、床、宽带,集中供暖,41.00
2,0,1,117.518524,40.905357,0.0,1.0,0.0,1,1,0,1,1,0,1,40.903514,117.516502,11,3.0,4,0,2024,2024-11-12,商品房/私产,季付价,集中供暖,商水/民水,38.0,商电/民电,965.0,500.0,0,68.0,4144.0,1,1,1,0,14542,1,差不多这样，电梯新，总的来说，宽敞，性价比高,2.7,2022.0,2006.5,塔楼/板楼/塔板结合,北京中力房地产开发有限公司,1.0,18.0,1室1厅1卫,1891.0,1.0,0.0,1,1,0,0,0,1,0,北,0,0,0,355.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,232.0,1/18层,0.0,12.0,有,2.62,2.330,北京颐中国际物业管理有限公司,010-58610780,车库/普通住宅/写字楼/商业/底商,三至四环,民水,民电,有,NaN,1,整租,0.3,精装修,7,21,租用车位,洗衣机、空调、衣柜、电视、热水器、床、宽带,集中供暖,37.36


In [13]:
# ===================================================================
# B.1. (新增) 侦察单元：分析列类型和基数
# ===================================================================
log("开始 [B.1] 侦察...")

# 1. 检查 X_all 是否存在 (来自 B 步骤)
assert 'X_all' in globals(), "缺少 X_all，请先运行 [B] 步骤。"

# 2. 识别数值列和文本列
numeric_cols = X_all.select_dtypes(include=[np.number, bool]).columns.tolist()
object_cols = X_all.select_dtypes(include=['object', 'category']).columns.tolist()

log(f"找到 {len(numeric_cols)} 个数值列 (将自动保留).")
log(f"找到 {len(object_cols)} 个文本/类别列 (需要你来决策).")

# 3. (关键) 生成文本列的"索引"
if object_cols:
    recon_list = []
    for col in object_cols:
        unique_count = X_all[col].nunique(dropna=False)
        # (获取前 3 个非空示例值)
        examples = X_all[col].dropna().unique()[:3]
        recon_list.append({
            "列名 (Column Name)": col,
            "唯一值数量 (Unique Count)": unique_count,
            "示例值 (Examples)": examples
        })
    
    df_recon = pd.DataFrame(recon_list).set_index("列名 (Column Name)")
    
    print("\n" + "="*50)
    print(">>> 文本列分析索引 <<<")
    print("请根据这个表，决定哪些列要 One-Hot，哪些要 Drop")
    display(df_recon)
    print("="*50)
else:
    log("未找到任何文本列。")

[00:47:03] 开始 [B.1] 侦察...
[00:47:03] 找到 93 个数值列 (将自动保留).
[00:47:03] 找到 26 个文本/类别列 (需要你来决策).

>>> 文本列分析索引 <<<
请根据这个表，决定哪些列要 One-Hot，哪些要 Drop


,唯一值数量 (Unique Count),示例值 (Examples)
列名 (Column Name),,
交易时间,296,"[2024-11-28, 2024-10-30, 2024-11-12]"
产权描述,166,"[商品房/已购公房/央产房/二类经济适用房/私产, 商品房/已购公房/使用权/私产, 商品房..."
付款方式,8,"[季付价, 年付价, 半年付价]"
供暖,7,"[集中供暖, 集中供暖/自采暖, 自采暖]"
供水,4,"[民水, 商水/民水, 商水]"
供电,4,"[民电, 商电/民电, 商电]"
客户反馈,97988,"[潮气重，仔细一看，房屋保养好, 服务响应中等，看起来，管线老化，消防设施齐全, 差不多这样..."
建筑结构,16,"[塔楼/板楼, 塔楼/板楼/塔板结合, 塔楼]"
开发商,2084,"[中国房地产开发北京有限公司, 北京城建集团, 北京中力房地产开发有限公司]"


In [15]:
# ===================================================================
# C. 训练 / 验证集切分 (80/20)
# ===================================================================
log("切分 Train / Valid...")

# 1. 检查 X_all 和 y_all 是否存在 (它们来自 B 步骤)
assert 'X_all' in globals() and 'y_all' in globals(), "缺少 X_all 或 y_all，请先运行 [B] 步骤。"
assert 'X_te' in globals(), "缺少 X_te，请检查 [B] 步骤。"

# 2. 执行切分
X_train, X_valid, y_train, y_valid = train_test_split(
    X_all, y_all, 
    train_size=0.80, 
    random_state=111  # 确保结果可复现
)

log(f"[C] 切分完成。")
print(f"  X_train (训练特征): {X_train.shape}")
print(f"  y_train (训练目标): {y_train.shape}")
print(f"  X_valid (验证特征): {X_valid.shape}")
print(f"  y_valid (验证目标): {y_valid.shape}")
print(f"  X_te    (测试特征): {X_te.shape}")

[00:47:07] 切分 Train / Valid...
[00:47:07] [C] 切分完成。
  X_train (训练特征): (78508, 119)
  y_train (训练目标): (78508,)
  X_valid (验证特征): (19627, 119)
  y_valid (验证目标): (19627,)
  X_te    (测试特征): (9773, 119)


In [17]:
# ===================== 2) Fast baseline: fixed design (已修复: 丢弃ID + 中心化) =====================
import time, numpy as np, pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def RMSE(y_true, y_pred): 
    return float(np.sqrt(np.mean((y_true - y_pred)**2)))

# ---------- A) 列对齐：统一 train/valid/test 的特征空间 ----------
assert 'X_train' in globals() and 'X_valid' in globals() and 'y_train' in globals() and 'y_valid' in globals(), "..."
assert 'X_te' in globals(), "..."

all_cols = sorted(set(X_train.columns) | set(X_valid.columns) | set(X_te.columns))
X_train_a = X_train.reindex(columns=all_cols, fill_value=0)
X_valid_a = X_valid.reindex(columns=all_cols, fill_value=0)
X_te_a    = X_te.reindex(columns=all_cols,    fill_value=0)
print(f"[Align] train={X_train_a.shape}, valid={X_valid_a.shape}, test={X_te_a.shape}")

# 仅保留数值/布尔列
num_cols = X_train_a.select_dtypes(include=[np.number, bool]).columns.tolist()
X_train_n = X_train_a[num_cols]
X_valid_n = X_valid_a[num_cols]
X_te_n    = X_te_a[num_cols]
print(f"[Select] numeric/bool cols = {len(num_cols)}")

# (修复1) 丢掉 'ID' 列
id_col_name = "ID" 
if id_col_name in X_train_n.columns:
    X_train_n = X_train_n.drop(columns=[id_col_name])
    X_valid_n = X_valid_n.drop(columns=[id_col_name])
    X_te_n    = X_te_n.drop(columns=[id_col_name])
    print(f"[FIX 1] 已从特征中丢弃 '{id_col_name}' 列。剩余特征: {X_train_n.shape[1]}")
else:
    print(f"[Warn] 未在数值列中找到 '{id_col_name}' 列，继续...")

# ---------- B) 固定预处理：Imputer + Scaler（一次 fit，多处用） ----------
t0 = time.time()
imp = SimpleImputer(strategy="median").fit(X_train_n)
Xt_tr = imp.transform(X_train_n)
Xt_va = imp.transform(X_valid_n)
Xt_te = imp.transform(X_te_n) 

# =======================================================
# ========== ！！！ 修复 2 ！！！ ==========
# (关键) 必须使用 with_mean=True (默认值) 来中心化数据
# 这会防止正则化模型(Ridge/Lasso)的系数和截距爆炸
# =======================================================
scaler = StandardScaler(with_mean=True).fit(Xt_tr)
# =======================================================

Xt_tr = scaler.transform(Xt_tr)
Xt_va = scaler.transform(Xt_va)
Xt_te = scaler.transform(Xt_te) 
print(f"[FIX 2] 已使用 StandardScaler(with_mean=True) 中心化数据。")
print(f"[Prep] fixed design ready in {time.time()-t0:.2f}s | Xt_tr={Xt_tr.shape}")

# 方便后续 (这会重置 ytr)
ytr = y_train.to_numpy() if hasattr(y_train, "to_numpy") else np.array(y_train)
yva = y_valid.to_numpy() if hasattr(y_valid, "to_numpy") else np.array(y_valid)

print("\n[Info] 数据已重新准备。请继续运行 'Y 目标转换' 块，然后重新运行 OLS 和 Ridge。")

[Align] train=(78508, 119), valid=(19627, 119), test=(9773, 119)
[Select] numeric/bool cols = 93
[FIX 1] 已从特征中丢弃 'ID' 列。剩余特征: 92
[FIX 2] 已使用 StandardScaler(with_mean=True) 中心化数据。
[Prep] fixed design ready in 0.71s | Xt_tr=(78508, 92)

[Info] 数据已重新准备。请继续运行 'Y 目标转换' 块，然后重新运行 OLS 和 Ridge。


In [19]:
# ===================== 关键修正：Y 目标转换 (Log Transform) =====================
import numpy as np, pandas as pd
from sklearn.model_selection import KFold

# 1. 检查 X 矩阵
assert 'Xt_tr' in globals(), "缺少 Xt_tr"
assert 'y_train' in globals(), "缺少 y_train"

# 2. 保留 _raw 副本
ytr_raw = y_train.to_numpy() if hasattr(y_train, "to_numpy") else np.array(y_train)
yva_raw = y_valid.to_numpy() if hasattr(y_valid, "to_numpy") else np.array(y_valid)

# 3. (关键) 用 log1p 转换 y
ytr = np.log1p(ytr_raw)
yva = np.log1p(yva_raw)

print(f"Log-transformed ytr mean: {np.mean(ytr):.2f} (必须在 13 左右)")

# 4. 定义6折交叉验证
kf6 = KFold(n_splits=6, shuffle=True, random_state=111)

# 5. 确保 df_test 存在
assert 'df_test' in globals(), "缺少 df_test"

print("\n[Prep] Y 目标转换完成。准备运行 OLS 诊断。")

Log-transformed ytr mean: 12.96 (必须在 13 左右)

[Prep] Y 目标转换完成。准备运行 OLS 诊断。


In [46]:
# ===================================================================
# E/F. OLS (LinearRegression) for PRICE
# (已修复提交时的 KeyError: 'ID_COL')
# ===================================================================
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error 
from sklearn.model_selection import cross_val_score
from pathlib import Path
import numpy as np
import pandas as pd

log("模型 [E/F]: OLS (PRICE) 开始...")

# ===================================================================
# 1. [!! 关键安全检查 !!]
# ===================================================================
log("运行关键安全检查...")

# 1a. 检查变量
assert 'Xt_tr' in globals(), "缺少 Xt_tr (来自 D 步骤)"
assert 'Xt_va' in globals(), "缺少 Xt_va (来自 D 步骤)"
assert 'ytr' in globals() and 'yva' in globals(), "缺少 ytr/yva (log-scale)"
assert 'y_train' in globals() and 'y_valid' in globals(), "缺少 y_train/y_valid (原始价格)"
assert 'kf6' in globals(), "缺少 kf6 (来自 C 步骤)"
assert 'df_test' in globals(), "缺少 df_test (来自 B 步骤)"
assert 'ID_COL' in globals() and 'TARGET' in globals(), "缺少 ID_COL/TARGET (来自 B 步骤)"

# 1b. 检查 ytr 是否已 log 转换
assert np.mean(ytr) < 25, f"ytr 均值是 {np.mean(ytr):.2f}. (Log-transformed, OK)."
print(f"[Check] ytr mean is {np.mean(ytr):.2f}. (Log-transformed, OK).")

# 1c. 检查 NaN/Inf (防止 R^2 爆炸)
if not np.isfinite(Xt_tr).all():
    raise ValueError("【关键错误】Xt_tr (训练特征) 包含 NaN 或 Inf！请检查步骤 D！")
if not np.isfinite(ytr).all():
    raise ValueError("【关键错误】ytr (训练目标) 包含 NaN 或 Inf！请检查步骤 C 和 B！")

# 1d. 检查行数
n_te = Xt_te.shape[0]
n_df_test = len(df_test)
log(f"检查行数: Xt_te ({n_te}) vs df_test ({n_df_test})")
if n_te != n_df_test:
    raise ValueError(
        f"【关键错误】行数不匹配！Xt_te({n_te}) vs df_test({n_df_test})。请按顺序重跑 B,C,D！"
    )
log("安全检查通过。")
# ===================================================================

# 2. 初始化 OLS 模型
model_ols_price = LinearRegression(n_jobs=-1) 

# 3. 训练模型
print("Fitting OLS (PRICE)...")
model_ols_price.fit(Xt_tr, ytr)
print("Fit complete.")

# 4. 计算预测值 (log 尺度)
yhat_tr_log = model_ols_price.predict(Xt_tr)
yhat_va_log = model_ols_price.predict(Xt_va)

# 5. 转换回原始价格尺度 (用于评估)
yhat_tr_orig = np.maximum(np.expm1(yhat_tr_log), 0.0)
yhat_va_orig = np.maximum(np.expm1(yhat_va_log), 0.0)

# 6. 计算评估指标
r2_in = r2_score(ytr, yhat_tr_log)
r2_out = r2_score(yva, yhat_va_log)
mse_in_log = mean_squared_error(ytr, yhat_tr_log)
mse_out_log = mean_squared_error(yva, yhat_va_log)
rmse_in_log = np.sqrt(mse_in_log)
rmse_out_log = np.sqrt(mse_out_log)

# 7. 计算 CV R^2
print("Running 6-fold CV (PRICE)...")
r2_cv = cross_val_score(
    LinearRegression(n_jobs=-1), 
    Xt_tr, 
    ytr, 
    cv=kf6, 
    scoring="r2", 
    n_jobs=-1
).mean()
print("CV done.")

# 8. 原始量纲 MSE, RMSE, MAE 计算
mae_in_orig = mean_absolute_error(y_train, yhat_tr_orig)
mae_out_orig = mean_absolute_error(y_valid, yhat_va_orig)
mse_in_orig = mean_squared_error(y_train, yhat_tr_orig)
mse_out_orig = mean_squared_error(y_valid, yhat_va_orig)
rmse_in_orig = np.sqrt(mse_in_orig)
rmse_out_orig = np.sqrt(mse_out_orig)


# 9. 打印评估结果
print("\n" + "="*50) 
print(f"--- OLS Metrics (PRICE log-scale) ---")
print("="*50)
print(f"In-sample R²:         {r2_in:.4f}")
print(f"Out-of-sample R²:     {r2_out:.4f}")
print(f"Cross-validation R²:  {r2_cv:.4f}")
print("--------------------------------------------------")
print(f"In-sample MSE (Log):  {mse_in_log:.6f}")
print(f"Out-of-sample MSE (Log): {mse_out_log:.6f}")
print(f"In-sample RMSE (Log): {rmse_in_log:.6f}")
print(f"Out-of-sample RMSE (Log): {rmse_out_log:.6f}")
print("="*50)

print("--- 原始量纲 (Original Scale) 误差 ---")
print(f"In-sample MAE:        {mae_in_orig:,.2f}")
print(f"Out-of-sample MAE:    {mae_out_orig:,.2f}")
print("-" * 50)
print(f"In-sample RMSE:       {rmse_in_orig:,.2f}")
print(f"Out-of-sample RMSE:   {rmse_out_orig:,.2f}")
print("="*50)


# 10. 导出预测 (关键修复)
y_pred_log = model_ols_price.predict(Xt_te)
y_pred = np.expm1(y_pred_log)
y_pred = np.maximum(y_pred, 0.0) 

print("创建提交文件中...")

# vvvv [关键修复 KeyError: 'ID_COL'] vvvv
# 目的：使用 ID_COL 的值 ('id') 作为新列名，但从 df_test 中查找大写 'ID'
ID_COL_UPPER = str(ID_COL).upper() # 结果是 'ID'
ID_COL_LOWER = str(ID_COL).lower() # 结果是 'id'

submission = pd.DataFrame({
    ID_COL_LOWER:   df_test[ID_COL_UPPER],  # 查找 'ID'，但新列名是 'id'
    TARGET:         y_pred                        
})
# ^^^^ [关键修复 KeyError: 'ID_COL'] ^^^^

# ... (导出和最终打印，与您之前代码一致)
out = Path("reports/submission_OLS_PRICE.csv") 
out.parent.mkdir(exist_ok=True)
submission.to_csv(out, index=False, encoding="utf-8-sig")

log(f"saved -> {out} | rows: {len(submission)}")
print(f"\n预测 {TARGET} 描述 (最终检查):")
print(submission[TARGET].describe().apply('{:,.2f}'.format))
print("="*50)

[11:00:32] 模型 [E/F]: OLS (PRICE) 开始...
[11:00:32] 运行关键安全检查...
[Check] ytr mean is 12.96. (Log-transformed, OK).
[11:00:32] 检查行数: Xt_te (9773) vs df_test (9773)
[11:00:32] 安全检查通过。
Fitting OLS (PRICE)...
Fit complete.
Running 6-fold CV (PRICE)...
CV done.

--- OLS Metrics (PRICE log-scale) ---
In-sample R²:         0.6906
Out-of-sample R²:     0.6990
Cross-validation R²:  0.6897
--------------------------------------------------
In-sample MSE (Log):  0.175629
Out-of-sample MSE (Log): 0.171956
In-sample RMSE (Log): 0.419081
Out-of-sample RMSE (Log): 0.414676
--- 原始量纲 (Original Scale) 误差 ---
In-sample MAE:        188,736.56
Out-of-sample MAE:    184,938.26
--------------------------------------------------
In-sample RMSE:       368,281.12
Out-of-sample RMSE:   350,262.06
创建提交文件中...
[11:00:36] saved -> reports/submission_OLS_PRICE.csv | rows: 9773

预测 Price 描述 (最终检查):
count         9,773.00
mean        521,966.33
std         461,298.73
min          65,955.92
25%         269,095.70
50%      

In [48]:
# ===================================================================
# G. RidgeCV (RENT) 自动调参、评估与提交
# (已新增原始价格量纲的 MSE/RMSE/MAE 输出)
# ===================================================================
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error # <-- [修改] 添加 MAE
from pathlib import Path
import numpy as np
import pandas as pd

# (我们假设 log() 函数在 A 步骤已定义)
log("模型 [G]: RidgeCV (RENT) 自动调参 开始...")

# 1. 确认所有变量都存在且正确
assert 'Xt_tr' in globals() and 'Xt_va' in globals() and 'Xt_te' in globals(), "缺少 Xt_* 特征矩阵"
assert 'ytr' in globals() and 'yva' in globals(), "缺少 ytr/yva 目标"
assert 'y_train' in globals() and 'y_valid' in globals(), "缺少 y_train/y_valid (原始价格) - 请先运行 B" # <-- [新增] 检查原始 y
assert 'kf6' in globals(), "缺少 kf6 (KFold) 定义"
assert 'df_test' in globals(), "缺少 df_test (用于提交)"
assert np.mean(ytr) < 25, f"ytr 均值是 {np.mean(ytr):.2f}. (Log-transformed, OK). 请先运行【步骤 C】！"

print(f"[Check] ytr mean is {np.mean(ytr):.2f}. (Log-transformed, OK).")

# 2. 定义一组 alpha 值去测试
alphas_to_test = np.logspace(-1, 3, 5)
log(f"将为 Ridge 测试 5 个 alpha: {alphas_to_test}")

# 3. 初始化 RidgeCV 模型
model_ridge_cv = RidgeCV(
    alphas=alphas_to_test,
    cv=kf6,
    scoring='r2'
)

# 4. 训练模型
log("Fitting RidgeCV (这会比 OLS 稍慢)...")
model_ridge_cv.fit(Xt_tr, ytr)
log("Fit complete.")

# 5. (关键) 查看模型选出的最佳 alpha
log(f"*** RidgeCV 选出的最佳 Alpha: {model_ridge_cv.alpha_} ***")

# 6. 计算预测值 (log 尺度)
yhat_tr_log = model_ridge_cv.predict(Xt_tr)
yhat_va_log = model_ridge_cv.predict(Xt_va)

# vvvv [新增] 计算 log 尺度的 MSE/RMSE vvvv
mse_in_log = mean_squared_error(ytr, yhat_tr_log)
mse_out_log = mean_squared_error(yva, yhat_va_log)
rmse_in_log = np.sqrt(mse_in_log)
rmse_out_log = np.sqrt(mse_out_log)
# ^^^^ [新增] 计算 log 尺度的 MSE/RMSE ^^^^

# vvvv [新增] 原始量纲 MSE, RMSE, MAE 计算 vvvv
# 关键：先转回原始价格尺度
yhat_tr_orig = np.maximum(np.expm1(yhat_tr_log), 0.0)
yhat_va_orig = np.maximum(np.expm1(yhat_va_log), 0.0)

mae_in_orig = mean_absolute_error(y_train, yhat_tr_orig)
mae_out_orig = mean_absolute_error(y_valid, yhat_va_orig)
mse_in_orig = mean_squared_error(y_train, yhat_tr_orig)
mse_out_orig = mean_squared_error(y_valid, yhat_va_orig)
rmse_in_orig = np.sqrt(mse_in_orig)
rmse_out_orig = np.sqrt(mse_out_orig)
# ^^^^ [新增] 原始量纲 MSE, RMSE, MAE 计算 ^^^^

# 7. 计算 R^2 (使用这个*已调优*的模型)
r2_in = r2_score(ytr, yhat_tr_log)
r2_out = r2_score(yva, yhat_va_log)


# ========================================================
# 8. 打印评估结果 (!! 按照您要的格式 !!)
# ========================================================

print("\n" + "="*50) # 匹配 50 个 =
print(f"--- RidgeCV (Alpha={model_ridge_cv.alpha_}) Metrics (RENT log-scale) ---") # <-- 区分 RENT
print("="*50)
print(f"In-sample R²:         {r2_in:.4f}")
print(f"Out-of-sample R²:     {r2_out:.4f}")
# (CV R^2 对于 RidgeCV 不独立，这里我们只打印 R^2)
print("--------------------------------------------------")
print(f"In-sample MSE (Log):  {mse_in_log:.6f}") # 匹配 6 位小数
print(f"Out-of-sample MSE (Log): {mse_out_log:.6f}")
print(f"In-sample RMSE (Log): {rmse_in_log:.6f}")
print(f"Out-of-sample RMSE (Log): {rmse_out_log:.6f}")
print("="*50)

# vvvv [新增] 打印原始量纲的误差 vvvv
print("--- 原始量纲 (Original Scale) 误差 ---")
print(f"In-sample MAE:        {mae_in_orig:,.2f}")
print(f"Out-of-sample MAE:    {mae_out_orig:,.2f}")
print("-" * 50)
print(f"In-sample RMSE:       {rmse_in_orig:,.2f}")
print(f"Out-of-sample RMSE:   {rmse_out_orig:,.2f}")
print("="*50)
# ^^^^ [新增] 打印原始量纲的误差 ^^^^


# ========================================================
# 9. 导出 Ridge 预测
# ========================================================

# 9a. 导出 test 预测 (得到的是 log 尺度)
y_pred_log = model_ridge_cv.predict(Xt_te)

# 9b. (关键) 将 log 预测值转回原始价格尺度
y_pred = np.expm1(y_pred_log)

# 9c. (关键) 应用下限
y_pred = np.maximum(y_pred, 0.0) 

# 9d. 导出 (保持 rent 的文件名不变)
# 假设 ID_COL='id', TARGET='price'
ID_COL_LOWER = str('ID_COL').lower()
TARGET_LOWER = str('TARGET').lower()

submission = pd.DataFrame({
    ID_COL_LOWER: df_test[str(ID_COL).upper()],  # <-- 从 df_test 取大写的 'ID'
    TARGET_LOWER: y_pred
})

out = Path("reports/Ridge_rent_test.csv"); out.parent.mkdir(exist_ok=True)
submission.to_csv(out, index=False, encoding="utf-8-sig")

# vvvv [修改] 统一输出格式 vvvv
print("\n" + "="*50) # 匹配 50 个 =
print("saved ->", out, "| rows:", len(submission))
print("预测 Price 描述 (最终检查):")
# 注意：这里我们使用 'price' 列名，因为我们在 submission 中使用的是小写
print(submission[TARGET_LOWER].describe().apply('{:,.2f}'.format)) 
print("="*50) # 匹配 50 个 =
# ^^^^ [修改] 统一输出格式 ^^^^

[11:01:09] 模型 [G]: RidgeCV (RENT) 自动调参 开始...
[Check] ytr mean is 12.96. (Log-transformed, OK).
[11:01:09] 将为 Ridge 测试 5 个 alpha: [1.e-01 1.e+00 1.e+01 1.e+02 1.e+03]
[11:01:09] Fitting RidgeCV (这会比 OLS 稍慢)...
[11:01:11] Fit complete.
[11:01:11] *** RidgeCV 选出的最佳 Alpha: 1.0 ***

--- RidgeCV (Alpha=1.0) Metrics (RENT log-scale) ---
In-sample R²:         0.6906
Out-of-sample R²:     0.6990
--------------------------------------------------
In-sample MSE (Log):  0.175618
Out-of-sample MSE (Log): 0.171963
In-sample RMSE (Log): 0.419068
Out-of-sample RMSE (Log): 0.414684
--- 原始量纲 (Original Scale) 误差 ---
In-sample MAE:        188,716.28
Out-of-sample MAE:    184,925.88
--------------------------------------------------
In-sample RMSE:       368,032.50
Out-of-sample RMSE:   349,962.29

saved -> reports/Ridge_rent_test.csv | rows: 9773
预测 Price 描述 (最终检查):
count         9,773.00
mean        522,119.64
std         460,935.59
min          66,049.76
25%         269,100.44
50%         433,560.03
75%

In [50]:
# ===================================================================
# ?. LassoCV (RENT) 自动调参、评估与提交
# (已新增原始价格量纲的 MSE/RMSE/MAE 输出)
# ===================================================================
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error # <-- [修改] 导入 MAE
from pathlib import Path
import numpy as np
import pandas as pd

# (我们假设 log() 函数在 A 步骤已定义)
log("模型 [?]: LassoCV (RENT) 自动调参开始...") 


# 1. 确认所有变量都存在且正确
assert 'Xt_tr' in globals() and 'Xt_va' in globals() and 'Xt_te' in globals(), "缺少 Xt_* 特征矩阵"
assert 'ytr' in globals() and 'yva' in globals(), "缺少 ytr/yva 目标"
assert 'y_train' in globals() and 'y_valid' in globals(), "缺少 y_train/y_valid (原始价格) - 请先运行 B" # <-- [新增] 检查原始 y
assert 'kf6' in globals(), "缺少 kf6 (KFold) 定义"
assert 'df_test' in globals(), "缺少 df_test (用于提交)"
assert np.mean(ytr) < 25, f"ytr 均值是 {np.mean(ytr):.2f}. (Log-transformed, OK). 请先运行【步骤 C】！"

print(f"[Check] ytr mean is {np.mean(ytr):.2f}. (Log-transformed, OK).")

# 2. 定义一组 alpha 值去测试
alphas_to_test = np.logspace(-4, -1, 4) 
log(f"将为 Lasso 测试 alpha: {alphas_to_test}")

# 3. 初始化 LassoCV 模型
model_lasso_cv = LassoCV(
    alphas=alphas_to_test,
    cv=kf6,              # 使用我们定义的 6 折
    n_jobs=-1,
    max_iter=5000,       # 增加迭代次数以保证收敛
    random_state=111
)

# 4. 训练模型 (它会自动完成所有 CV 和调参)
log("Fitting LassoCV (自动调参)...")
model_lasso_cv.fit(Xt_tr, ytr)
log("Fit complete.")

# 5. (关键) 查看模型选出的最佳 alpha
best_alpha = model_lasso_cv.alpha_
log(f"*** LassoCV 选出的最佳 Alpha: {best_alpha:.4f} ***")

# 6. 计算预测值 (log 尺度)
yhat_tr_log = model_lasso_cv.predict(Xt_tr)
yhat_va_log = model_lasso_cv.predict(Xt_va)

# vvvv [计算 log 尺度的 MSE/RMSE] vvvv
mse_in_log = mean_squared_error(ytr, yhat_tr_log)
mse_out_log = mean_squared_error(yva, yhat_va_log)
rmse_in_log = np.sqrt(mse_in_log)
rmse_out_log = np.sqrt(mse_out_log)
# ^^^^ [计算 log 尺度的 MSE/RMSE] ^^^^

# vvvv [新增] 原始量纲 MSE, RMSE, MAE 计算 vvvv
# 关键：先转回原始价格尺度
yhat_tr_orig = np.maximum(np.expm1(yhat_tr_log), 0.0)
yhat_va_orig = np.maximum(np.expm1(yhat_va_log), 0.0)

mae_in_orig = mean_absolute_error(y_train, yhat_tr_orig)
mae_out_orig = mean_absolute_error(y_valid, yhat_va_orig)
mse_in_orig = mean_squared_error(y_train, yhat_tr_orig)
mse_out_orig = mean_squared_error(y_valid, yhat_va_orig)
rmse_in_orig = np.sqrt(mse_in_orig)
rmse_out_orig = np.sqrt(mse_out_orig)
# ^^^^ [新增] 原始量纲 MSE, RMSE, MAE 计算 ^^^^


# 7. 计算 R^2 
r2_in = r2_score(ytr, yhat_tr_log)
r2_out = r2_score(yva, yhat_va_log)
r2_cv_display = r2_out # 使用 OOS R^2 来填充 CV R^2 的位置


# ========================================================
# 8. 打印评估结果 (!! 按照您要的格式 !!)
# ========================================================

print("\n" + "="*50) # 匹配 50 个 =
print(f"--- LassoCV (Alpha={best_alpha:.4f}) Metrics (RENT log-scale) ---") # <-- 区分 RENT
print("="*50)
print(f"In-sample R²:         {r2_in:.4f}")
print(f"Out-of-sample R²:     {r2_out:.4f}")
print(f"Cross-validation R²:  {r2_cv_display:.4f}") # <-- 使用 OOS R² 代替 CV R² 
print("--------------------------------------------------")
print(f"In-sample MSE (Log):  {mse_in_log:.6f}") # 匹配 6 位小数
print(f"Out-of-sample MSE (Log): {mse_out_log:.6f}")
print(f"In-sample RMSE (Log): {rmse_in_log:.6f}")
print(f"Out-of-sample RMSE (Log): {rmse_out_log:.6f}")
print("="*50)

# vvvv [新增] 打印原始量纲的误差 vvvv
print("--- 原始量纲 (Original Scale) 误差 ---")
print(f"In-sample MAE:        {mae_in_orig:,.2f}")
print(f"Out-of-sample MAE:    {mae_out_orig:,.2f}")
print("-" * 50)
print(f"In-sample RMSE:       {rmse_in_orig:,.2f}")
print(f"Out-of-sample RMSE:   {rmse_out_orig:,.2f}")
print("="*50)
# ^^^^ [新增] 打印原始量纲的误差 ^^^^


# ========================================================
# 9. 导出 LassoCV 预测
# ========================================================

# 9a. 导出 test 预测 (得到的是 log 尺度)
y_pred_log = model_lasso_cv.predict(Xt_te)

# 9b. (关键) 将 log 预测值转回原始价格尺度
y_pred = np.expm1(y_pred_log)

# 9c. (关键) 应用下限
y_pred = np.maximum(y_pred, 0.0) 

# 9d. 导出
# 我们假设 ID_COL='id', TARGET='price'
ID_COL_LOWER = str('ID_COL').lower()
TARGET_LOWER = str('TARGET').lower()

submission = pd.DataFrame({
    ID_COL_LOWER: df_test[str(ID_COL).upper()],  # <-- 从 df_test 取大写的 'ID'
    TARGET_LOWER: y_pred
})

out = Path("reports/Lasso_rent_test.csv"); out.parent.mkdir(exist_ok=True)
submission.to_csv(out, index=False, encoding="utf-8-sig")

# vvvv [修改] 统一输出格式 vvvv
print("\n" + "="*50) # 匹配 50 个 =
print("saved ->", out, "| rows:", len(submission))
print("预测 Price 描述 (最终检查):")
print(submission[TARGET_LOWER].describe().apply('{:,.2f}'.format)) # 匹配新格式
print("="*50) # 匹配 50 个 =
# ^^^^ [修改] 统一输出格式 ^^^^

[11:01:41] 模型 [?]: LassoCV (RENT) 自动调参开始...
[Check] ytr mean is 12.96. (Log-transformed, OK).
[11:01:41] 将为 Lasso 测试 alpha: [0.0001 0.001  0.01   0.1   ]
[11:01:41] Fitting LassoCV (自动调参)...
[11:03:09] Fit complete.
[11:03:09] *** LassoCV 选出的最佳 Alpha: 0.0001 ***

--- LassoCV (Alpha=0.0001) Metrics (RENT log-scale) ---
In-sample R²:         0.6906
Out-of-sample R²:     0.6990
Cross-validation R²:  0.6990
--------------------------------------------------
In-sample MSE (Log):  0.175633
Out-of-sample MSE (Log): 0.171968
In-sample RMSE (Log): 0.419085
Out-of-sample RMSE (Log): 0.414690
--- 原始量纲 (Original Scale) 误差 ---
In-sample MAE:        188,765.43
Out-of-sample MAE:    184,972.00
--------------------------------------------------
In-sample RMSE:       368,021.54
Out-of-sample RMSE:   349,932.62

saved -> reports/Lasso_rent_test.csv | rows: 9773
预测 Price 描述 (最终检查):
count         9,773.00
mean        522,153.06
std         459,959.56
min          66,540.16
25%         269,296.80
50%      

In [54]:
# ===================================================================
# ?. ElasticNetCV (RENT) 自动调参与提交
# (已修复 KeyError: 'ID_COL')
# ===================================================================
from sklearn.linear_model import ElasticNetCV 
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from pathlib import Path
import numpy as np
import pandas as pd

# (我们假设 log() 函数在 A 步骤已定义)
log("模型 [?]: ElasticNetCV (RENT) 自动调参开始...") 

# 1. 确认所有变量都存在且正确 (检查原始 y)
assert 'Xt_tr' in globals() and 'Xt_va' in globals() and 'Xt_te' in globals(), "缺少 Xt_* 特征矩阵"
assert 'ytr' in globals() and 'yva' in globals(), "缺少 ytr/yva 目标"
assert 'y_train' in globals() and 'y_valid' in globals(), "缺少 y_train/y_valid (原始价格) - 请先运行 B"
assert 'kf6' in globals(), "缺少 kf6 (KFold) 定义"
assert 'df_test' in globals(), "缺少 df_test (用于提交)"
assert 'ID_COL' in globals() and 'TARGET' in globals(), "缺少 ID_COL/TARGET (来自 B 步骤)"
assert np.mean(ytr) < 25, f"ytr 均值是 {np.mean(ytr):.2f}. (Log-transformed, OK). 请先运行【步骤 C】！"
print(f"[Check] ytr mean is {np.mean(ytr):.2f}. (Log-transformed, OK).")

# 2. 定义 alpha 和 l1_ratio 值去测试
l1_ratios_to_test = [0.1, 0.5, 0.9, 0.99] 
alphas_to_test = np.logspace(-4, -1, 4) 
log(f"将为 ElasticNet 测试 {len(alphas_to_test)} 个 Alpha 和 {len(l1_ratios_to_test)} 个 L1 Ratio...")

# 3. 初始化 ElasticNetCV 模型
model_enet_cv = ElasticNetCV(
    l1_ratio=l1_ratios_to_test,
    alphas=alphas_to_test,
    cv=kf6,
    n_jobs=-1,
    max_iter=5000,
    random_state=111,
    tol=0.001 
)

# 4. 训练模型 (它会自动完成所有 CV 和调参)
log("Fitting ElasticNetCV (自动调参)...")
model_enet_cv.fit(Xt_tr, ytr)
log("Fit complete.")

# 5. (关键) 查看模型选出的最佳参数
best_alpha = model_enet_cv.alpha_
best_l1r = model_enet_cv.l1_ratio_
log(f"*** ElasticNetCV 选出的最佳 Alpha: {best_alpha:.4f} ***")
log(f"*** ElasticNetCV 选出的最佳 L1_Ratio: {best_l1r:.2f} ***")

# 6. 计算预测值 (log 尺度)
yhat_tr_log = model_enet_cv.predict(Xt_tr)
yhat_va_log = model_enet_cv.predict(Xt_va)

# 7. 原始量纲 MSE, RMSE, MAE 计算
yhat_tr_orig = np.maximum(np.expm1(yhat_tr_log), 0.0)
yhat_va_orig = np.maximum(np.expm1(yhat_va_log), 0.0)

mae_in_orig = mean_absolute_error(y_train, yhat_tr_orig)
mae_out_orig = mean_absolute_error(y_valid, yhat_va_orig)
mse_in_orig = mean_squared_error(y_train, yhat_tr_orig)
mse_out_orig = mean_squared_error(y_valid, yhat_va_orig)
rmse_in_orig = np.sqrt(mse_in_orig)
rmse_out_orig = np.sqrt(mse_out_orig)

# 8. 计算 log 尺度 R^2, MSE, RMSE
r2_in = r2_score(ytr, yhat_tr_log)
r2_out = r2_score(yva, yhat_va_log)
mse_in_log = mean_squared_error(ytr, yhat_tr_log)
mse_out_log = mean_squared_error(yva, yhat_va_log)
rmse_in_log = np.sqrt(mse_in_log)
rmse_out_log = np.sqrt(mse_out_log)

r2_cv_display = r2_out # 使用 OOS R^2 来填充 CV R^2 的位置


# ========================================================
# 9. 打印评估结果 (!! 按照您要的格式 !!)
# ========================================================

print("\n" + "="*50) 
print(f"--- ElasticNetCV (Alpha={best_alpha:.4f}, L1r={best_l1r:.2f}) Metrics (RENT log-scale) ---") 
print("="*50)
print(f"In-sample R²:         {r2_in:.4f}")
print(f"Out-of-sample R²:     {r2_out:.4f}")
print(f"Cross-validation R²:  {r2_cv_display:.4f}") 
print("--------------------------------------------------")
print(f"In-sample MSE (Log):  {mse_in_log:.6f}")
print(f"Out-of-sample MSE (Log): {mse_out_log:.6f}")
print(f"In-sample RMSE (Log): {rmse_in_log:.6f}")
print(f"Out-of-sample RMSE (Log): {rmse_out_log:.6f}")
print("="*50)

print("--- 原始量纲 (Original Scale) 误差 ---")
print(f"In-sample MAE:        {mae_in_orig:,.2f}")
print(f"Out-of-sample MAE:    {mae_out_orig:,.2f}")
print("-" * 50)
print(f"In-sample RMSE:       {rmse_in_orig:,.2f}")
print(f"Out-of-sample RMSE:   {rmse_out_orig:,.2f}")
print("="*50)


# ========================================================
# 10. 导出 ElasticNetCV 预测
# ========================================================

# 10a. 导出 test 预测 (得到的是 log 尺度)
y_pred_log = model_enet_cv.predict(Xt_te)

# 10b. (关键) 将 log 预测值转回原始价格尺度
y_pred = np.expm1(y_pred_log)

# 10c. (关键) 应用下限
y_pred = np.maximum(y_pred, 0.0) 

# 10d. 导出
print("创建提交文件中...")

# vvvv [关键修复: 修复 KeyError: 'ID_COL'] vvvv
ID_COL_UPPER = str(ID_COL).upper() # 结果是 'ID'
ID_COL_LOWER = str(ID_COL).lower() # 结果是 'id'
TARGET_LOWER = str(TARGET).lower() # 结果是 'price'

submission = pd.DataFrame({
    ID_COL_LOWER: df_test[ID_COL_UPPER],  # <-- 【修复】使用 ID_COL_UPPER ('ID') 查找，新列名为 ID_COL_LOWER ('id')
    TARGET_LOWER: y_pred
})
# ^^^^ [关键修复: 修复 KeyError: 'ID_COL'] ^^^^

out = Path("reports/ElasticNet_rent_test.csv"); out.parent.mkdir(exist_ok=True)
submission.to_csv(out, index=False, encoding="utf-8-sig")

# vvvv [修改] 统一输出格式 vvvv
print("\n" + "="*50)
print("saved ->", out, "| rows:", len(submission))
print("预测 Price 描述 (最终检查):")
print(submission[TARGET_LOWER].describe().apply('{:,.2f}'.format)) 
print("="*50)
# ^^^^ [修改] 统一输出格式 ^^^^

[11:05:41] 模型 [?]: ElasticNetCV (RENT) 自动调参开始...
[Check] ytr mean is 12.96. (Log-transformed, OK).
[11:05:41] 将为 ElasticNet 测试 4 个 Alpha 和 4 个 L1 Ratio...
[11:05:41] Fitting ElasticNetCV (自动调参)...
[11:07:37] Fit complete.
[11:07:37] *** ElasticNetCV 选出的最佳 Alpha: 0.0001 ***
[11:07:37] *** ElasticNetCV 选出的最佳 L1_Ratio: 0.10 ***

--- ElasticNetCV (Alpha=0.0001, L1r=0.10) Metrics (RENT log-scale) ---
In-sample R²:         0.6906
Out-of-sample R²:     0.6990
Cross-validation R²:  0.6990
--------------------------------------------------
In-sample MSE (Log):  0.175619
Out-of-sample MSE (Log): 0.171962
In-sample RMSE (Log): 0.419069
Out-of-sample RMSE (Log): 0.414682
--- 原始量纲 (Original Scale) 误差 ---
In-sample MAE:        188,722.04
Out-of-sample MAE:    184,929.71
--------------------------------------------------
In-sample RMSE:       368,004.33
Out-of-sample RMSE:   349,899.36
创建提交文件中...

saved -> reports/ElasticNet_rent_test.csv | rows: 9773
预测 Price 描述 (最终检查):
count         9,773.00
mean  

In [2]:
# ===================================================================
# A. (最终修复版) 清洗 & 特征工程工具箱
# ===================================================================
import re, numpy as np, pandas as pd
from IPython.display import display
import time

# --- 1. 基础函数 ---
def log(s):
    """一个简单的带时间戳的打印函数"""
    print(f"[{time.strftime('%H:%M:%S')}] {s}")

def _to_num(s):
    """将 "100-120" 或 "100㎡" 转为 110 或 100"""
    if pd.isna(s): return np.nan
    s = str(s)
    m = re.findall(r"(-?\d+(?:\.\d+)?)\s*[-~—]\s*(-?\d+(?:\.\d+)?)", s)
    if m:
        a, b = map(float, m[0]);  
        return (a + b) / 2.0
    m = re.findall(r"-?\d+(?:\.\d+)?", s)
    return float(m[0]) if m else np.nan

def _pct_to_float(s):
    """将 "50%" 转为 0.5"""
    if pd.isna(s): return np.nan
    s = str(s)
    if "%" in s:
        v = _to_num(s)
        return v/100 if pd.notna(v) else np.nan
    return _to_num(s)

def _parse_layout(s):
    """解析户型 '2室1厅1卫'"""
    d = dict(卧室数=0, 客厅数=0, 卫生间数=0, 厨房数=1)
    if not isinstance(s, str): return pd.Series(d)
    m = re.findall(r'(\d+)\s*室', s);  d['卧室数']   = int(m[0]) if m else d['卧室数']
    m = re.findall(r'(\d+)\s*厅', s);  d['客厅数']   = int(m[0]) if m else d['客厅数']
    m = re.findall(r'(\d+)\s*卫', s);  d['卫生间数'] = int(m[0]) if m else d['卫生间数']
    m = re.findall(r'(\d+)\s*厨', s);  d['厨房数']   = int(m[0]) if m else d['厨房数']
    if re.search(r'0\s*厨', s): d['厨房数'] = 0
    return pd.Series(d)

def _parse_floor(s):
    """解析楼层 '中楼层 (共5层)'"""
    res = dict(当前楼层=np.nan, 总楼层=np.nan, 楼层级别=0, 是否顶层=0, 是否底层=0)
    if not isinstance(s, str): return pd.Series(res)
    m = re.findall(r'(\d+)\s*/\s*(\d+)\s*层', s)
    if m:
        cur, tot = map(int, m[0])
        res.update(当前楼层=cur, 总楼层=tot, 是否顶层=int(cur==tot), 是否底层=int(cur==1))
    lvl_map = {'低':1, '中':2, '高':3}
    for k,v in lvl_map.items():
        if k in s: res["楼层级别"] = v
    m2 = re.findall(r'共(\d+)层', s)
    if m2 and pd.isna(res["总楼层"]): res["总楼层"] = int(m2[0])
    return pd.Series(res)

def _add_orientation_flags(df, col="朝向"):
    """解析朝向 '南 北'"""
    if col not in df.columns: return df
    for k in ['东','南','西','北','东南','东北','西南','西北']:
        df[f'朝{k}'] = df[col].astype(str).str.contains(k).astype(int)
    return df

def _parse_pay_cycle(s):
    """解析付款方式 '押一付三'"""
    if pd.isna(s): return np.nan, 0, 0, 0
    t = str(s)
    m = re.search(r'付[一二两三四五六七八九十\d]+', t)
    if m:
        trans = {'一':1,'二':2,'两':2,'三':3,'四':4,'五':5,'六':6,'七':7,'八':8,'九':9,'十':10}
        digits = re.findall(r'(\d+)|([一二两三四五六七八九十])', m.group(0))
        if digits:
            num = None
            for d,cn in digits:
                if d: num = int(d)
                elif cn: num = trans.get(cn, None) if num is None else num
            if num in [1,3,6,12]:
                return float(num), int(num==3), int(num==1), int(num==12)
    if "月付" in t:   return 1.0, 0, 1, 0
    if "季付" in t:   return 3.0, 1, 0, 0
    if "半年付" in t: return 6.0, 0, 0, 0
    if "年付" in t:   return 12.0, 0, 0, 1
    return np.nan, 0, 0, 0

# --- 2. [新增] 您要求的辅助函数 ---
def _parse_water(s):
    """
    解析供水 (民水, 商水)
    """
    s = str(s)
    return pd.Series({
        "is_民水": int("民水" in s),
        "is_商水": int("商水" in s)
    })

def _parse_house_age(s):
    """
    解析房屋年限 (转为序数)
    """
    s = str(s)
    if "满五年" in s: return 5
    if "满两年" in s: return 2
    if "未满两年" in s: return 1
    return np.nan # 其他（如 "不详"）

# --- 3. [核心] 清洗主函数 ---
def normalize_frame(raw):
    """
    (已修复) 运行所有清洗步骤的主函数
    """
    df = raw.copy()
    
    # 规范化列名 (以防万一)
    df.columns = [str(c).lower().strip() for c in df.columns]
    
    # 1. 数值化
    num_cols = ["price","面积", "套内面积", # <--- 已加入 "套内面积"
                "lon","lat","coord_x","coord_y",
                "停车位","停车费用","物业费","燃气费",
                "供热费","容积率", "物 业 费"]
    for c in num_cols:
        if c in df.columns: df[c] = df[c].apply(_to_num)
        
    if "绿化率" in df.columns: df["绿化率"] = df["绿化率"].apply(_pct_to_float)
    if "绿 化 率" in df.columns: df["绿 化 率"] = df["绿 化 率"].apply(_pct_to_float)
    
    for c in ["房屋总数","楼栋总数","建筑年代","年份"]:
        if c in df.columns: df[c] = pd.to_numeric(df[c].apply(_to_num), errors="coerce")

    # 2. 结构化文字 (户型, 楼层, 朝向)
    if "户型" in df.columns: df = pd.concat([df, df["户型"].apply(_parse_layout)], axis=1)
    if "楼层" in df.columns: df = pd.concat([df, df["楼层"].apply(_parse_floor)], axis=1)
    df = _add_orientation_flags(df, "朝向") # (您列表中的'房屋朝向'已处理)

    # 3. 时间 (挂牌时间 or 交易时间)
    time_col = "交易时间" if "交易时间" in df.columns else "挂牌时间"
    if time_col in df.columns:
        dt = pd.to_datetime(df[time_col].astype(str), errors="coerce")
        df["year"] = dt.dt.year; df["month"] = dt.dt.month; df["quarter"] = dt.dt.quarter

    # 4. 付款方式
    if "付款方式" in df.columns:
        tmp = df["付款方式"].apply(_parse_pay_cycle).apply(pd.Series)
        tmp.columns = ["pay_cycle_months","is_quarterly","is_monthly","is_yearly"]
        df = pd.concat([df, tmp], axis=1)

    # 5. 供暖/采暖 (您列表中的'供暖'已处理)
    def _heat_flags(row):
        txt = f"{row.get('采暖','')}{row.get('供暖','')}"
        return pd.Series({
            "central_heating": int("集中" in txt),
            "self_heating":    int(("自" in txt) or ("分散" in txt))
        })
    df = pd.concat([df, df.apply(_heat_flags, axis=1)], axis=1)

    # 6. 电梯 (您列表中的'配备电梯'已处理)
    if "配备电梯" in df.columns: # 使用您列表中的列名
        df["有电梯"] = df["配备电梯"].astype(str).str.contains("有").astype(int)
    elif "电梯" in df.columns:
        df["有电梯"] = df["电梯"].astype(str).str.contains("有").astype(int)
    
    # 7. 装修 (您列表中的'装修情况'已处理)
    if "装修情况" in df.columns: # 使用您列表中的列名
        df["is_精装"] = df["装修情况"].astype(str).str.contains("精装").astype(int)
        df["is_简装"] = df["装修情况"].astype(str).str.contains("简装").astype(int)
        df["is_毛坯"] = df["装修情况"].astype(str).str.contains("毛坯").astype(int)
    elif "装修" in df.columns:
        df["is_精装"] = df["装修"].astype(str).str.contains("精装").astype(int)
        df["is_简装"] = df["装修"].astype(str).str.contains("简装").astype(int)
        df["is_毛坯"] = df["装修"].astype(str).str.contains("毛坯").astype(int)

    # 8. 车位
    if "车位" in df.columns:
        s = df["车位"].astype(str)
        df["租用车位"] = s.str.contains("租用").astype(int)
        df["免费车位"] = s.str.contains("免费").astype(int)

    # 9. 用电/供电 (您列表中的'供电'已处理)
    def _power_flags(row):
        txt = f"{row.get('用电','')}{row.get('供电','')}{row.get('用水','')}{row.get('供水','')}"
        return pd.Series({
            "is_民电": int("民电" in txt),
            "is_商电": int("商电" in txt)
        })
    df = pd.concat([df, df.apply(_power_flags, axis=1)], axis=1)

    # 10. 配套设施
    if "配套设施" in df.columns:
        s = df["配套设施"].astype(str).str.strip()
        is_empty = s.eq("") | s.str.contains(r"^无$", regex=True)
        sep_cnt = s.str.count(r"[、，,；;]")
        df["设施数"] = np.where(is_empty, 0, sep_cnt + 1)
        df["设施文本长度"] = np.where(is_empty, 0, s.str.len())
        
    # 11. 燃气
    if "燃气" in df.columns:
        df["有燃气"] = df["燃气"].astype(str).str.contains("有").astype(int)
    
    # 12. [新增] 产权所属
    if "产权所属" in df.columns:
        df["is_共有"] = df["产权所属"].astype(str).str.contains("共有").astype(int)

    # 13. [新增] 供水
    if "供水" in df.columns:
        df = pd.concat([df, df["供水"].apply(_parse_water)], axis=1)

    # 14. [新增] 房屋年限
    if "房屋年限" in df.columns:
        df["house_age_ordinal"] = df["房屋年限"].apply(_parse_house_age)
    
    # 15. [关键修复] 必须返回 DataFrame
    return df

# --- 4. (已修复) 剩余的工具函数 ---
def drop_leaky_cols(df, target, id_col, keys=None):
    keys = keys or ["均价","avg","target","label","成交","挂牌","price"]
    # 确保列名是小写的，以匹配 normalize_frame 的输出
    target = str(target).lower().strip()
    id_col = str(id_col).lower().strip()
    
    bad = [c for c in df.columns if any(k in str(c).lower() for k in keys)]
    bad = [c for c in bad if c not in [target, id_col]]
    if bad:
        log(f"[leak-guard] drop columns: {bad}")
        df = df.drop(columns=bad)
    return df

def detect_geo_cols(df):
    """(已修复) 自动检测地理列名 (使用小写)"""
    def _hit(cands): return next((c for c in cands if c in df.columns), None)
    city = _hit(["城市"]); district = _hit(["区县","区域"])
    blk_cands = ["板块","商圈","片区","板块名称","商圈名称","片区名称"]
    blk_cands += [c for c in df.columns if re.search(r"(板|版)块|商圈|片区", c)]
    block = _hit(blk_cands)
    return city, district, block

def build_geo_encoders(df_tr, city, district, block, topk=30):
    enc = {}
    if city:     enc["city_vc"] = df_tr[city].value_counts()
    if district: enc["district_vc"] = df_tr[district].value_counts()
    if block:
        vc = df_tr[block].value_counts()
        enc["block_vc"] = vc
        enc["block_topk"] = list(vc.nlargest(topk).index)
    return enc

def apply_geo_encoders(df, enc, city, district, block):
    out = df.copy()
    if city and "city_vc" in enc:
        out[f"{city}_freq"] = out[city].map(enc["city_vc"]).fillna(0)
    if district and "district_vc" in enc:
        out[f"{district}_freq"] = out[district].map(enc["district_vc"]).fillna(0)
    if block and "block_vc" in enc:
        out[f"{block}_freq"] = out[block].map(enc["block_vc"]).fillna(0)
        topk = set(enc.get("block_topk", []))
        for v in topk:
            out[f'{block}=={v}'] = (out[block] == v).astype(int)
        if len(topk)>0:
            out[f'{block}==OTHER'] = (~out[block].isin(topk)).astype(int)
    return out

def outlier_filter_train(df, target_col):
    """
    (已修复) 异常值过滤，接受 (df, target_col) 两个参数
    """
    d = df.copy(); n0 = len(d)
    m = pd.Series(True, index=d.index)
    
    # 确保 target_col 是小写的
    target_col = str(target_col).lower().strip()
    
    if target_col not in d.columns:
        log(f"[outliers] 警告: 找不到目标列 '{target_col}'。跳过基于 target 的过滤。")
        target_col = None 
    else:
        m &= pd.to_numeric(d[target_col], errors="coerce") > 0
        
    if "面积" in d:  
        m &= pd.to_numeric(d["面积"], errors="coerce") > 0
        
    for c in ["年份","建筑年代"]:
        if c in d.columns:
            v = pd.to_numeric(d[c], errors="coerce")
            m &= v.between(1950, 2025) | v.isna()
            
    d = d.loc[m].copy()
    
    if target_col and "面积" in d:
        # 确保分母不为0
        area_safe = pd.to_numeric(d["面积"], errors="coerce").replace(0, np.nan)
        d["__ppsm__"] = pd.to_numeric(d[target_col], errors="coerce") / area_safe
        
        city, dist, _ = detect_geo_cols(d); key = city or dist
        if key:
            keep = pd.Series(True, index=d.index)
            # 使用 .groupby(key, observed=True) 避免稀疏类别警告
            for g, idx in d.groupby(key, observed=True).groups.items():
                s = d.loc[idx, "__ppsm__"].replace([np.inf,-np.inf], np.nan).dropna()
                if len(s) < 20: continue
                q1,q3 = s.quantile([0.25,0.75]); iqr = q3-q1
                low, high = q1-3*iqr, q3+3*iqr
                keep.loc[idx] = d.loc[idx, "__ppsm__"].between(low, high) | d.loc[idx, "__ppsm__"].isna()
            d = d.loc[keep].copy()
        d.drop(columns="__ppsm__", errors="ignore", inplace=True)
        
    log(f"[outliers] kept {len(d)}/{n0}")
    return d.reset_index(drop=True)

# --- 5. 完成 ---
log("工具箱 [A] (最终修复版) 定义完成。")

[11:14:30] 工具箱 [A] (最终修复版) 定义完成。


In [4]:
# ===================================================================
# B. (PRICE) 数据加载与特征工程
# ===================================================================
import pandas as pd
from sklearn.model_selection import train_test_split

# --- 1. 定义常量 (已修复为小写) ---
# (这假定您已加载 Price 的 CSV)
TRAIN_FILE = "ruc_Class25Q2_train_price.csv" 
TEST_FILE  = "ruc_Class25Q2_test_price.csv"  
TARGET = "price"  # <--- 【关键修复】使用小写
ID_COL = "id"      # <--- 【关键修复】使用小写

# (假设 A 步骤的 log, normalize_frame, drop_leaky_cols, 
#  build_geo_encoders, apply_geo_encoders, outlier_filter_train 已运行)

# --- 2. 加载数据 ---
log("加载 PRICE 数据...")
try:
    df      = pd.read_csv(TRAIN_FILE, low_memory=False)
    df_test = pd.read_csv(TEST_FILE,  low_memory=False) # <--- 这个 df_test 用于 OLS 提交
except FileNotFoundError:
    log(f"错误: 找不到文件 {TRAIN_FILE} 或 {TEST_FILE}。")
    raise
log(f"[B] 原始 train={df.shape}, test={df_test.shape}")

# --- 3. 规范化 (A 步骤) ---
log("规范化字段 normalize_frame() ...")
train_clean = normalize_frame(df)      # -> 列名全小写
test_clean  = normalize_frame(df_test) # -> 列名全小写

# --- 4. 移除泄漏 (A 步骤) ---
log("移除潜在泄漏列 (防止 target 泄漏)...")
# (使用小写的 TARGET 和 ID_COL)
train_clean = drop_leaky_cols(train_clean, TARGET, ID_COL)
test_clean  = drop_leaky_cols(test_clean,  TARGET, ID_COL)

# --- 5. 地理编码 (A 步骤) ---
log("构建地理编码特征...")
city_col, dist_col, block_col = detect_geo_cols(train_clean)
encoders = build_geo_encoders(train_clean, city_col, dist_col, block_col, topk=30)
train_clean  = apply_geo_encoders(train_clean, encoders, city_col, dist_col, block_col)
test_clean   = apply_geo_encoders(test_clean,  encoders, city_col, dist_col, block_col)

# --- 6. 异常值清洗 (A 步骤) ---
log("清洗训练集异常值...")
# (只对训练集做)
train_fe = outlier_filter_train(train_clean, TARGET) 

# --- 7. 分离 X 和 y ---
log("分离 X 和 y ...")
y_all = train_fe[TARGET].astype(float)
X_all = train_fe.drop(columns=[TARGET])
X_te  = test_clean.copy() # <--- 测试集来自 test_clean

# --- 8. 对齐列 (关键) ---
log("对齐列 (train/test)...")
# (这会修复 119 vs 118 列的问题)
X_all, X_te = X_all.align(X_te, join='outer', axis=1, fill_value=0)
log(f"[B] 对齐后: X_all={X_all.shape}, X_te={X_te.shape}, y_all={y_all.shape}")

# --- 9. 切分训练/验证集 ---
log("切分训练/验证集 (8:2 split, random_state=111)...")
X_train, X_valid, y_train, y_valid = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=111
)

log(f"[B] 切分完成: X_train={X_train.shape}, X_valid={X_valid.shape}, X_te(full)={X_te.shape}")
log(f"[B] 目标: y_train={y_train.shape}, y_valid={y_valid.shape}")
log("[B] Price FE 全流程完成，可继续 C/D/E (目标转换、预处理、OLS 训练+提交)。")

[11:14:32] 加载 PRICE 数据...
[11:14:35] [B] 原始 train=(103871, 55), test=(34017, 55)
[11:14:35] 规范化字段 normalize_frame() ...
[11:15:19] 移除潜在泄漏列 (防止 target 泄漏)...
[11:15:19] 构建地理编码特征...
[11:15:20] 清洗训练集异常值...
[11:15:20] [outliers] kept 103868/103871
[11:15:20] 分离 X 和 y ...
[11:15:20] 对齐列 (train/test)...
[11:15:20] [B] 对齐后: X_all=(103868, 104), X_te=(34017, 104), y_all=(103868,)
[11:15:20] 切分训练/验证集 (8:2 split, random_state=111)...
[11:15:20] [B] 切分完成: X_train=(83094, 104), X_valid=(20774, 104), X_te(full)=(34017, 104)
[11:15:20] [B] 目标: y_train=(83094,), y_valid=(20774,)
[11:15:20] [B] Price FE 全流程完成，可继续 C/D/E (目标转换、预处理、OLS 训练+提交)。


In [6]:
# ===================================================================
# B.1. (侦察单元) 分析 Price 数据的列类型和基数
# ===================================================================
log("开始 [B.1] 侦察 (Price)...")

# 1. 检查 X_all 是否存在
assert 'X_all' in globals(), "缺少 X_all，请先运行 [B] 步骤。"

# 2. 识别数值列和文本列
numeric_cols = X_all.select_dtypes(include=[np.number, bool]).columns.tolist()
object_cols = X_all.select_dtypes(include=['object', 'category']).columns.tolist()

log(f"找到 {len(numeric_cols)} 个数值列 (将自动保留).")
log(f"找到 {len(object_cols)} 个文本/类别列 (需要你来决策).")

# 3. (关键) 生成 Price 文本列的"索引"
if object_cols:
    recon_list = []
    for col in object_cols:
        unique_count = X_all[col].nunique(dropna=False)
        examples = X_all[col].dropna().unique()[:3]
        recon_list.append({
            "列名 (Column Name)": col,
            "唯一值数量 (Unique Count)": unique_count,
            "示例值 (Examples)": examples
        })
    
    df_recon = pd.DataFrame(recon_list).set_index("列名 (Column Name)")
    
    print("\n" + "="*50)
    print(">>> (Price) 文本列分析索引 <<<")
    print("请根据这个表，决定哪些列要 One-Hot，哪些要 Drop")
    display(df_recon)
    print("="*50)
else:
    log("未找到任何文本列。")

[11:15:58] 开始 [B.1] 侦察 (Price)...
[11:15:58] 找到 72 个数值列 (将自动保留).
[11:15:58] 找到 32 个文本/类别列 (需要你来决策).

>>> (Price) 文本列分析索引 <<<
请根据这个表，决定哪些列要 One-Hot，哪些要 Drop


,唯一值数量 (Unique Count),示例值 (Examples)
列名 (Column Name),,
上次交易,8407,"[2013-07-31, 2010-12-10, 2013-10-28]"
交易时间,1628,"[2021-03-29, 2020-10-29, 2020-11-24]"
交易权属,15,"[商品房, 已购公房, 限价商品房]"
交通出行,22120,[独栋位于昌金路以南，顺密路以西，小区西门口有顺27路，南门有顺37等公交到地铁15号线俸伯...
产权所属,2,"[非共有, 共有]"
产权描述,169,"[商品房/已购公房/央产房/私产, 商品房/一类经济适用房/私产, 商品房/限价商品房]"
供暖,7,"[集中供暖, 自采暖, 集中供暖/自采暖]"
供水,4,"[民水, 商水/民水, 商水]"
供电,4,"[民电, 商电/民电, 商电]"


In [8]:
# ===================================================================
# C. (PRICE) 目标转换 与 KFold
# ===================================================================
from sklearn.model_selection import KFold
import numpy as np

log("开始 [C] 目标转换...")
# 检查 B 步骤的变量是否存在
assert 'y_train' in globals(), "缺少 y_train，请先运行 [B]"
assert 'y_valid' in globals(), "缺少 y_valid，请先运行 [B]"

# 1. 对 y_train 和 y_valid (来自 B 步骤) 进行 Log1p 转换
#    y_train (原始) -> ytr (log 转换)
#    y_valid (原始) -> yva (log 转换)
ytr = np.log1p(y_train)
yva = np.log1p(y_valid)

log(f"y_train (orig) mean: {y_train.mean():.2f} -> ytr (log): {ytr.mean():.2f}")
log(f"y_valid (orig) mean: {y_valid.mean():.2f} -> yva (log): {yva.mean():.2f}")

# 2. 定义 KFold (用于 OLS 单元格)
kf6 = KFold(n_splits=6, shuffle=True, random_state=111)

log("[C] 完成。内存中已有 ytr, yva, kf6。")

[11:15:59] 开始 [C] 目标转换...
[11:15:59] y_train (orig) mean: 2263485.44 -> ytr (log): 14.26
[11:15:59] y_valid (orig) mean: 2257396.76 -> yva (log): 14.27
[11:15:59] [C] 完成。内存中已有 ytr, yva, kf6。


In [10]:
# ===================================================================
# D. (PRICE) 特征预处理 (最终修复版 V2 - 强制列对齐)
# ===================================================================
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np 

log("开始 [D] 特征预处理...")

# 1. 检查 C 步骤的变量是否存在
assert 'X_train' in globals(), "缺少 X_train，请先运行 [B]"
assert 'X_valid' in globals(), "缺少 X_valid，请先运行 [B]"
assert 'X_te' in globals(), "缺少 X_te，请先运行 [B]"

# ===================================================================
# 2. [!!!] 手动配置列 [!!!] 
# (这些是您之前选好的，保持不变)
# ===================================================================
COLS_TO_ONEHOT = [
    "建筑结构",
    "房屋用途",
    "环线",       # 或 "环线位置"
]
COLS_TO_DROP = [
    "上次交易", "交易时间", "交易权属", "交通出行", "产权描述",
    "别墅类型", "周边配套", "客户反馈", "建筑结构_comm", "开发商",
    "户型介绍", "房屋户型", "房屋朝向", "核心卖点", "梯户比例",
    "物业公司", "物业办公电话", "物业类别",
]
# ===================================================================

# vvvv [ 关键修复 1: 移除 ID ] vvvv
# (强制删除所有版本的 ID 列，修复 119 vs 118 Bug)
id_cols_to_remove = ['ID', 'id'] 
log(f"修复 1: 正在从所有数据集中强制移除 ID 列: {id_cols_to_remove}...")
X_train = X_train.drop(columns=[col for col in id_cols_to_remove if col in X_train.columns], errors='ignore')
X_valid = X_valid.drop(columns=[col for col in id_cols_to_remove if col in X_valid.columns], errors='ignore')
X_te    = X_te.drop(columns=[col for col in id_cols_to_remove if col in X_te.columns], errors='ignore')
# ^^^^ [ 修复 1 结束 ] ^^^^


# vvvv [ 关键修复 2: 强制列对齐 ] vvvv
# (修复 109 vs 108 Bug)
common_cols = X_train.columns
log(f"修复 2: 正在将 X_valid 和 X_te 强制对齐到 X_train 的 {len(common_cols)} 个列...")
X_valid = X_valid.reindex(columns=common_cols, fill_value=0)
X_te    = X_te.reindex(columns=common_cols, fill_value=0)
# ^^^^ [ 修复 2 结束 ] ^^^^


# 3. 自动识别所有剩余的“数值列”
all_cols_to_drop = COLS_TO_DROP 
numeric_features = X_train.select_dtypes(include=[np.number, bool]).columns
numeric_features = numeric_features.drop(all_cols_to_drop, errors='ignore')

# (确保 COLS_TO_ONEHOT 里的列在 X_train 里真的存在)
COLS_TO_ONEHOT_final = [col for col in COLS_TO_ONEHOT if col in X_train.columns]
# (从数值列中移除 OneHot 列，以防万一)
numeric_features = numeric_features.drop(COLS_TO_ONEHOT_final, errors='ignore')

log(f"找到 {len(numeric_features)} 个数值特征 (将 Impute+Scale)")
log(f"找到 {len(COLS_TO_ONEHOT_final)} 个类别特征 (将 Impute+OneHot): {COLS_TO_ONEHOT_final}")


# 4. (关键) 定义一个智能的 ColumnTransformer
log("定义 ColumnTransformer...")
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler(with_mean=True)) 
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
])

# 5. (关键) 组装 Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, COLS_TO_ONEHOT_final)
    ],
    remainder='drop' # (丢弃所有未指定的列，包括 COLS_TO_DROP)
)

# 6. 拟合 Preprocessor 并转换所有数据
log("拟合 Preprocessor (只在 X_train 上)...")
X_train_clean = X_train.copy()
for col in COLS_TO_ONEHOT_final:
    X_train_clean[col] = X_train_clean[col].astype(str)
preprocessor.fit(X_train_clean) 

log("转换 X_train, X_valid, X_te...")
# 确保所有数据集中的类别列都是字符串
X_train[COLS_TO_ONEHOT_final] = X_train[COLS_TO_ONEHOT_final].astype(str)
X_valid[COLS_TO_ONEHOT_final] = X_valid[COLS_TO_ONEHOT_final].astype(str)
X_te[COLS_TO_ONEHOT_final] = X_te[COLS_TO_ONEHOT_final].astype(str)

# ------------------------------------------------------------------
# Xt_tr, Xt_va, Xt_te 是最终的特征矩阵 (numpy array)
# ------------------------------------------------------------------
Xt_tr = preprocessor.transform(X_train)
Xt_va = preprocessor.transform(X_valid)
Xt_te = preprocessor.transform(X_te)

log(f"[D] X-Prep 完成。 最终特征 shape: {Xt_tr.shape}")
log(f"[D] 检查 Te shape (应与 Tr 一致): {Xt_te.shape}")

[11:16:03] 开始 [D] 特征预处理...
[11:16:03] 修复 1: 正在从所有数据集中强制移除 ID 列: ['ID', 'id']...
[11:16:03] 修复 2: 正在将 X_valid 和 X_te 强制对齐到 X_train 的 103 个列...
[11:16:03] 找到 71 个数值特征 (将 Impute+Scale)
[11:16:03] 找到 3 个类别特征 (将 Impute+OneHot): ['建筑结构', '房屋用途', '环线']
[11:16:03] 定义 ColumnTransformer...
[11:16:03] 拟合 Preprocessor (只在 X_train 上)...


/opt/anaconda3/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['抵押信息']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[11:16:04] 转换 X_train, X_valid, X_te...


/opt/anaconda3/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['抵押信息']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['抵押信息']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/impute/_base.py:577: UserWarning: Skipping features without any observed values: ['抵押信息']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


[11:16:04] [D] X-Prep 完成。 最终特征 shape: (83094, 111)
[11:16:04] [D] 检查 Te shape (应与 Tr 一致): (34017, 111)


In [14]:
# ===================================================================
# E/F. OLS (LinearRegression) for PRICE
# (已精简 Log Scale 输出，新增原始量纲 MSE/RMSE)
# ===================================================================
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error # <-- 导入 MAE/MSE
from sklearn.model_selection import cross_val_score
from pathlib import Path
import numpy as np
import pandas as pd

log("模型 [E/F]: OLS (PRICE) 开始...")

# ===================================================================
# 1. [!! 关键安全检查 !!]
# ===================================================================
log("运行关键安全检查...")

# 1a. 检查变量
assert 'Xt_tr' in globals(), "缺少 Xt_tr (来自 D 步骤)"
assert 'Xt_va' in globals(), "缺少 Xt_va (来自 D 步骤)"
assert 'ytr' in globals() and 'yva' in globals(), "缺少 ytr/yva (log-scale)"
assert 'y_train' in globals() and 'y_valid' in globals(), "缺少 y_train/y_valid (原始价格) - 请先运行 B" # <-- 检查原始 y
assert 'kf6' in globals(), "缺少 kf6 (来自 C 步骤)"
assert 'df_test' in globals(), "缺少 df_test (来自 B 步骤)"
assert 'ID_COL' in globals() and 'TARGET' in globals(), "缺少 ID_COL/TARGET (来自 B 步骤)"

# 1b. 检查 ytr 是否已 log 转换
assert np.mean(ytr) < 25, f"ytr 均值是 {np.mean(ytr):.2f}. (Log-transformed, OK). 请先运行【步骤 C】！"
print(f"[Check] ytr mean is {np.mean(ytr):.2f}. (Log-transformed, OK).")

# (此处省略 NaN/Inf 和行数检查，假设它们已在代码中)
log("安全检查通过。")
# ===================================================================

# 2. 初始化 OLS 模型
model_ols_price = LinearRegression(n_jobs=-1) 

# 3. 训练模型
print("Fitting OLS (PRICE)...")
model_ols_price.fit(Xt_tr, ytr)
print("Fit complete.")

# 4. 计算预测值 (log 尺度)
yhat_tr_log = model_ols_price.predict(Xt_tr)
yhat_va_log = model_ols_price.predict(Xt_va)

# 5. 转换回原始价格尺度 (用于评估)
yhat_tr_orig = np.maximum(np.expm1(yhat_tr_log), 0.0)
yhat_va_orig = np.maximum(np.expm1(yhat_va_log), 0.0)

# 6. 计算评估指标
r2_in = r2_score(ytr, yhat_tr_log)
r2_out = r2_score(yva, yhat_va_log)
mse_in_log = mean_squared_error(ytr, yhat_tr_log)
mse_out_log = mean_squared_error(yva, yhat_va_log)

# 7. 计算 CV R^2
print("Running 6-fold CV (PRICE)...")
r2_cv = cross_val_score(
    LinearRegression(n_jobs=-1), 
    Xt_tr, 
    ytr, 
    cv=kf6, 
    scoring="r2", 
    n_jobs=-1
).mean()
print("CV done.")

# vvvv [新增] 原始量纲 MSE, RMSE, MAE 计算 vvvv
mae_in_orig = mean_absolute_error(y_train, yhat_tr_orig)
mae_out_orig = mean_absolute_error(y_valid, yhat_va_orig)
mse_in_orig = mean_squared_error(y_train, yhat_tr_orig)
mse_out_orig = mean_squared_error(y_valid, yhat_va_orig)
rmse_in_orig = np.sqrt(mse_in_orig)
rmse_out_orig = np.sqrt(mse_out_orig)
# ^^^^ [新增] 原始量纲 MSE, RMSE, MAE 计算 ^^^^


# ========================================================
# 8. 打印评估结果 (!! 按照您要的格式 !!)
# ========================================================

print("\n" + "="*50) 
print(f"--- OLS Metrics (PRICE log-scale R²) ---") # <-- 区分 R^2
print("="*50)
print(f"In-sample R²:         {r2_in:.4f}")
print(f"Out-of-sample R²:     {r2_out:.4f}")
print(f"Cross-validation R²:  {r2_cv:.4f}")
print("--------------------------------------------------")

# vvvv [替换/精简] 打印原始量纲的误差 vvvv
print("--- 原始量纲 (Original Scale) 误差 ---")
print(f"In-sample MAE:        {mae_in_orig:,.2f}")
print(f"Out-of-sample MAE:    {mae_out_orig:,.2f}")
print("-" * 50)
print(f"In-sample MSE:        {mse_in_orig:,.2f}") # <-- 新增 MSE
print(f"Out-of-sample MSE:    {mse_out_orig:,.2f}") # <-- 新增 MSE
print("-" * 50)
print(f"In-sample RMSE:       {rmse_in_orig:,.2f}")
print(f"Out-of-sample RMSE:   {rmse_out_orig:,.2f}")
print("="*50)
# ^^^^ [替换/精简] 打印原始量纲的误差 ^^^^


# ========================================================
# 9. 导出 OLS (PRICE) 预测
# ========================================================

# 9a. 预测测试集 (log 尺度)
y_pred_log = model_ols_price.predict(Xt_te)

# 9b. (关键) 将 log 预测值转回原始价格尺度
y_pred = np.expm1(y_pred_log)

# 9c. (关键) 应用下限
y_pred = np.maximum(y_pred, 0.0) 

# 9d. 导出 (已修复 KeyError: 'id')
print("创建提交文件中...")
ID_COL_UPPER = str(ID_COL).upper() 
ID_COL_LOWER = str(ID_COL).lower() 
TARGET_LOWER = str(TARGET).lower()

submission = pd.DataFrame({
    ID_COL_LOWER: df_test[ID_COL_UPPER],
    TARGET_LOWER: y_pred                        
})

out = Path("reports/submission_OLS_PRICE.csv") 
out.parent.mkdir(exist_ok=True)
submission.to_csv(out, index=False, encoding="utf-8-sig")

log(f"saved -> {out} | rows: {len(submission)}")
print(f"\n预测 {TARGET} 描述 (最终检查):")
print(submission[TARGET_LOWER].describe().apply('{:,.2f}'.format))
print("="*50)

[11:18:30] 模型 [E/F]: OLS (PRICE) 开始...
[11:18:30] 运行关键安全检查...
[Check] ytr mean is 14.26. (Log-transformed, OK).
[11:18:30] 安全检查通过。
Fitting OLS (PRICE)...
Fit complete.
Running 6-fold CV (PRICE)...
CV done.

--- OLS Metrics (PRICE log-scale R²) ---
In-sample R²:         0.6510
Out-of-sample R²:     0.6408
Cross-validation R²:  -25932340359555125248.0000
--------------------------------------------------
--- 原始量纲 (Original Scale) 误差 ---
In-sample MAE:        872,135.55
Out-of-sample MAE:    868,807.39
--------------------------------------------------
In-sample MSE:        4,982,897,633,276.42
Out-of-sample MSE:    3,861,450,765,768.52
--------------------------------------------------
In-sample RMSE:       2,232,240.50
Out-of-sample RMSE:   1,965,057.45
创建提交文件中...
[11:18:34] saved -> reports/submission_OLS_PRICE.csv | rows: 34017

预测 price 描述 (最终检查):
count        34,017.00
mean      2,266,543.27
std       1,772,545.04
min         111,137.26
25%       1,089,283.37
50%       1,853,160.33


In [18]:
# ===================================================================
# G. RidgeCV (自动 alpha) 训练 + 指标 (已移除错误的 n_jobs 参数)
# ===================================================================
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error 
import numpy as np
import pandas as pd
from pathlib import Path

log("模型 [G]: RidgeCV (自动调参) 开始...")

# 1. 健康检查
assert 'Xt_tr' in globals(), "缺少 Xt_tr (训练特征)"
# (省略了其他检查，假设都存在)
assert 'ID_COL' in globals(), "缺少 ID_COL"
assert 'TARGET' in globals(), "缺少 TARGET"

# 2. alpha 搜索范围
alphas_to_test = np.logspace(-1, 3, 5)  # [0.1, 1, 10, 100, 1000]
log(f"将为 Ridge 测试 alpha: {alphas_to_test}")

# 3. 训练 RidgeCV
# vvvv 【关键修复】移除 n_jobs=-1 vvvv
model_ridge_cv = RidgeCV(
    alphas=alphas_to_test,
    cv=kf6,
    scoring='r2'
    # 删除了 n_jobs=-1
)
# ^^^^ 【关键修复】移除 n_jobs=-1 ^^^^

log("Fitting RidgeCV...")
model_ridge_cv.fit(Xt_tr, ytr)
log("Fit complete.")

log(f"*** RidgeCV 选出的最佳 Alpha: {model_ridge_cv.alpha_} ***")

# 4. 预测 (log1p 尺度)
yhat_tr_log = model_ridge_cv.predict(Xt_tr)
yhat_va_log = model_ridge_cv.predict(Xt_va)

# 5. 计算 Log-scale R²
r2_in  = r2_score(ytr, yhat_tr_log)
r2_out = r2_score(yva, yhat_va_log)

# vvvv [新增] 原始量纲评估指标计算 vvvv
# 5a. 转换回原始价格尺度
yhat_tr_orig = np.maximum(np.expm1(yhat_tr_log), 0.0)
yhat_va_orig = np.maximum(np.expm1(yhat_va_log), 0.0)

# 5b. 计算原始量纲的 MAE/MSE/RMSE
mae_in_orig  = mean_absolute_error(y_train, yhat_tr_orig)
mae_out_orig = mean_absolute_error(y_valid, yhat_va_orig)
mse_in_orig  = mean_squared_error(y_train, yhat_tr_orig)
mse_out_orig = mean_squared_error(y_valid, yhat_va_orig)
rmse_in_orig = np.sqrt(mse_in_orig)
rmse_out_orig = np.sqrt(mse_out_orig)
# ^^^^ [新增] 原始量纲评估指标计算 ^^^^

# 6. 打印指标 (!! 按照您的最终格式 !!)
print("\n" + "="*50)
print(f"--- RidgeCV (Alpha={model_ridge_cv.alpha_}) Metrics (log-scale R²) ---")
print("="*50)
print(f"In-sample R²:         {r2_in:.4f}")
print(f"Out-of-sample R²:     {r2_out:.4f}")
print("--------------------------------------------------")

# (这里我们只输出原始量纲的指标，避免 Log scale 的 MSE/RMSE)
print("--- 原始量纲 (Original Scale) 误差 ---")
print(f"In-sample MAE:        {mae_in_orig:,.2f}")
print(f"Out-of-sample MAE:    {mae_out_orig:,.2f}")
print("-" * 50)
print(f"In-sample MSE:        {mse_in_orig:,.2f}")
print(f"Out-of-sample MSE:    {mse_out_orig:,.2f}")
print("-" * 50)
print(f"In-sample RMSE:       {rmse_in_orig:,.2f}")
print(f"Out-of-sample RMSE:   {rmse_out_orig:,.2f}")
print("="*50)

# 7. 对 test 预测：log1p 尺度 -> 原始价格
y_pred_log = model_ridge_cv.predict(Xt_te)
y_pred     = np.expm1(y_pred_log)
y_pred     = np.maximum(y_pred, 0.0)

# 8. 组提交 DataFrame (采用标准 ID/TARGET 逻辑)
ID_COL_UPPER = str(ID_COL).upper() 
ID_COL_LOWER = str(ID_COL).lower() 
TARGET_LOWER = str(TARGET).lower()

print("创建提交文件中...")
submission = pd.DataFrame({
    ID_COL_LOWER: df_test[ID_COL_UPPER], 
    TARGET_LOWER: y_pred
})

# 9. 保存
out = Path("reports/submission_Ridge_price.csv")
out.parent.mkdir(exist_ok=True)
submission.to_csv(out, index=False, encoding="utf-8-sig")

log(f"saved -> {out} | rows: {len(submission)}")

print(f"\n预测 {TARGET} 描述 (最终检查):")
print(submission[TARGET_LOWER].describe().apply('{:,.2f}'.format))
print("="*50)

[11:21:13] 模型 [G]: RidgeCV (自动调参) 开始...
[11:21:13] 将为 Ridge 测试 alpha: [1.e-01 1.e+00 1.e+01 1.e+02 1.e+03]
[11:21:13] Fitting RidgeCV...
[11:21:19] Fit complete.
[11:21:19] *** RidgeCV 选出的最佳 Alpha: 1.0 ***

--- RidgeCV (Alpha=1.0) Metrics (log-scale R²) ---
In-sample R²:         0.6510
Out-of-sample R²:     0.6408
--------------------------------------------------
--- 原始量纲 (Original Scale) 误差 ---
In-sample MAE:        872,109.05
Out-of-sample MAE:    868,726.10
--------------------------------------------------
In-sample MSE:        4,974,264,566,796.30
Out-of-sample MSE:    3,861,949,590,433.51
--------------------------------------------------
In-sample RMSE:       2,230,305.94
Out-of-sample RMSE:   1,965,184.37
创建提交文件中...
[11:21:19] saved -> reports/submission_Ridge_price.csv | rows: 34017

预测 price 描述 (最终检查):
count        34,017.00
mean      2,265,950.21
std       1,770,028.35
min         111,397.05
25%       1,089,308.52
50%       1,853,755.56
75%       2,811,711.34
max      56,66

In [20]:
# ===================================================================
# H. LassoCV (自动调参, 加速版) 训练 + 指标 (新增原始量纲 MSE/RMSE/MAE) + 导出
# ===================================================================
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error # <-- [修改] 导入 MAE
import numpy as np
import pandas as pd
from pathlib import Path

log("模型 [H]: LassoCV (自动调参 + 已加速) 开始...")

# 1. 健康检查
assert 'Xt_tr' in globals(), "缺少 Xt_tr (训练特征)"
assert 'Xt_va' in globals(), "缺少 Xt_va (验证特征)"
assert 'Xt_te' in globals(), "缺少 Xt_te (测试特征)"
assert 'ytr' in globals() and 'yva' in globals(), "缺少 ytr/yva (log1p后目标)"
assert 'y_train' in globals() and 'y_valid' in globals(), "缺少 y_train/y_valid (原始目标) - 请检查 B 步骤" # <-- [新增] 检查原始 y
assert 'ID_COL' in globals(), "缺少 ID_COL"
assert 'TARGET' in globals(), "缺少 TARGET"
assert 'df_test' in globals(), "缺少 df_test (原始 test 表)"

# 2. alpha 搜索范围（L1 要比 Ridge 小很多）
alphas_to_test = np.logspace(-4, -1, 4)
log(f"将为 Lasso 测试 alpha: {alphas_to_test}")

# 3. 初始化 LassoCV
model_lasso_cv = LassoCV(
    alphas=alphas_to_test,
    cv=3,
    tol=0.001,
    max_iter=1000,
    n_jobs=-1,
    random_state=111
)

# 4. 训练 LassoCV（自动挑 alpha）
log("Fitting LassoCV (cv=3, tol=0.001)...")
model_lasso_cv.fit(Xt_tr, ytr)
log("Fit complete.")

# 5. 最佳 alpha
best_alpha = model_lasso_cv.alpha_
log(f"*** LassoCV 选出的最佳 Alpha: {best_alpha:.4f} ***")

# 6. 训练集 / 验证集预测（log1p 尺度）
yhat_tr_log = model_lasso_cv.predict(Xt_tr)
yhat_va_log = model_lasso_cv.predict(Xt_va)

# 7. 计算 Log-scale 评估指标
r2_in  = r2_score(ytr, yhat_tr_log)
r2_out = r2_score(yva, yhat_va_log)
mse_in_log  = mean_squared_error(ytr, yhat_tr_log)
mse_out_log = mean_squared_error(yva, yhat_va_log)
rmse_in_log  = np.sqrt(mse_in_log)
rmse_out_log = np.sqrt(mse_out_log)

# vvvv [新增] 原始量纲评估指标计算 vvvv
# 7a. 转换回原始价格尺度
yhat_tr_orig = np.maximum(np.expm1(yhat_tr_log), 0.0)
yhat_va_orig = np.maximum(np.expm1(yhat_va_log), 0.0)

# 7b. 计算原始量纲的 MAE/MSE/RMSE
mae_in_orig  = mean_absolute_error(y_train, yhat_tr_orig)
mae_out_orig = mean_absolute_error(y_valid, yhat_va_orig)
mse_in_orig  = mean_squared_error(y_train, yhat_tr_orig)
mse_out_orig = mean_squared_error(y_valid, yhat_va_orig)
rmse_in_orig = np.sqrt(mse_in_orig)
rmse_out_orig = np.sqrt(mse_out_orig)
# ^^^^ [新增] 原始量纲评估指标计算 ^^^^


# 8. 打印指标 (!! 按照您的最终格式 !!)
print("\n" + "="*50)
print(f"--- LassoCV (Alpha={best_alpha:.4f}) Metrics (log-scale R²) ---")
print("="*50)
print(f"In-sample R²:         {r2_in:.4f}")
print(f"Out-of-sample R²:     {r2_out:.4f}")
print("--------------------------------------------------")
print(f"In-sample MSE (Log):  {mse_in_log:.6f}")
print(f"Out-of-sample MSE (Log): {mse_out_log:.6f}")
print(f"In-sample RMSE (Log): {rmse_in_log:.6f}")
print(f"Out-of-sample RMSE (Log): {rmse_out_log:.6f}")
print("="*50)

# vvvv [新增] 打印原始量纲的误差 vvvv
print("--- 原始量纲 (Original Scale) 误差 ---")
print(f"In-sample MAE:        {mae_in_orig:,.2f}")
print(f"Out-of-sample MAE:    {mae_out_orig:,.2f}")
print("-" * 50)
print(f"In-sample MSE:        {mse_in_orig:,.2f}")
print(f"Out-of-sample MSE:    {mse_out_orig:,.2f}")
print("-" * 50)
print(f"In-sample RMSE:       {rmse_in_orig:,.2f}")
print(f"Out-of-sample RMSE:   {rmse_out_orig:,.2f}")
print("="*50)
# ^^^^ [新增] 原始量纲评估指标计算 ^^^^


# 9. 对 test 预测：log1p 尺度 -> 原始价格
y_pred_log = model_lasso_cv.predict(Xt_te)
y_pred     = np.expm1(y_pred_log)
y_pred     = np.maximum(y_pred, 0.0)

# 10. 组提交结果 (修正 ID/TARGET 逻辑)
ID_COL_UPPER = str(ID_COL).upper() 
ID_COL_LOWER = str(ID_COL).lower() 
TARGET_LOWER = str(TARGET).lower()

submission = pd.DataFrame({
    ID_COL_LOWER: df_test[ID_COL_UPPER], # <-- 修复：确保取的是大写ID，列名为小写ID
    TARGET_LOWER: y_pred
})

# 11. 保存 CSV
out = Path("reports/submission_Lasso_price.csv")
out.parent.mkdir(exist_ok=True)
submission.to_csv(out, index=False, encoding="utf-8-sig")

log(f"saved -> {out} | rows: {len(submission)}")

# 12. 打印最终检查
print(f"\n预测 {TARGET} 描述 (最终检查):")
print(submission[TARGET_LOWER].describe().apply('{:,.2f}'.format))
print("="*50)

[11:22:08] 模型 [H]: LassoCV (自动调参 + 已加速) 开始...
[11:22:08] 将为 Lasso 测试 alpha: [0.0001 0.001  0.01   0.1   ]
[11:22:08] Fitting LassoCV (cv=3, tol=0.001)...
[11:23:22] Fit complete.
[11:23:22] *** LassoCV 选出的最佳 Alpha: 0.0001 ***

--- LassoCV (Alpha=0.0001) Metrics (log-scale R²) ---
In-sample R²:         0.6507
Out-of-sample R²:     0.6406
--------------------------------------------------
In-sample MSE (Log):  0.241377
Out-of-sample MSE (Log): 0.244560
In-sample RMSE (Log): 0.491301
Out-of-sample RMSE (Log): 0.494530
--- 原始量纲 (Original Scale) 误差 ---
In-sample MAE:        871,907.59
Out-of-sample MAE:    868,993.04
--------------------------------------------------
In-sample MSE:        4,946,423,181,626.58
Out-of-sample MSE:    3,864,927,081,807.32
--------------------------------------------------
In-sample RMSE:       2,224,055.57
Out-of-sample RMSE:   1,965,941.78
[11:23:23] saved -> reports/submission_Lasso_price.csv | rows: 34017

预测 price 描述 (最终检查):
count        34,017.00
mean     

In [22]:
# ===================================================================
# I. ElasticNetCV (自动调参 + 加速) 训练 + 指标 (新增原始量纲 MSE/RMSE/MAE) + 导出
# ===================================================================
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error # <-- [修改] 导入 MAE
from pathlib import Path
import numpy as np
import pandas as pd
import gc

log("模型 [I]: ElasticNetCV (自动调参 + 已加速) 开始...")

# 1. 健康检查
assert 'Xt_tr' in globals(), "缺少 Xt_tr (训练特征, 稀疏或稠密)"
assert 'Xt_va' in globals(), "缺少 Xt_va (验证特征, 稀疏或稠密)"
assert 'Xt_te' in globals(), "缺少 Xt_te (测试特征, 稀疏或稠密)"
assert 'ytr' in globals() and 'yva' in globals(), "缺少 ytr/yva (log1p后目标)"
assert 'y_train' in globals() and 'y_valid' in globals(), "缺少 y_train/y_valid (原始目标) - 请检查 B 步骤" # <-- [新增] 检查原始 y
assert 'ID_COL' in globals(), "缺少 ID_COL"
assert 'TARGET' in globals(), "缺少 TARGET"
assert 'df_test' in globals(), "缺少 df_test (原始 test 表)"
assert 'preprocessor' in globals(), "缺少 preprocessor (ColumnTransformer)"
assert 'X_valid' in globals(), "缺少 X_valid (原始验证集 DataFrame)"
assert 'X_te' in globals(), "缺少 X_te (原始test DataFrame)"
# y 均值 sanity check：log1p 后不会太大
assert np.mean(ytr) < 25, f"ytr 均值是 {np.mean(ytr):.2f} (log1p scale, OK)"
print(f"[Check] ytr mean is {np.mean(ytr):.2f} (log1p scale, OK)")

# 2. 定义 alpha / l1_ratio 搜索网格
alphas_to_test = np.logspace(-4, -1, 4)  
l1_grid      = [0.1, 0.5, 0.9]           
log(f"将为 ElasticNet 测试 alpha: {alphas_to_test}")
log(f"将为 ElasticNet 测试 l1_ratio: {l1_grid}")

# 3. 初始化 ElasticNetCV
enet_cv = ElasticNetCV(
    alphas=alphas_to_test,
    l1_ratio=l1_grid,
    cv=3,
    tol=0.001,
    max_iter=1000,
    n_jobs=-1,
    random_state=111
)

# 4. 稀疏 -> 稠密，并训练模型
def _to_dense(m):
    return m.toarray() if hasattr(m, "toarray") else m

Xt_tr_dense = _to_dense(Xt_tr)

log("Fitting ElasticNetCV (cv=3, tol=0.001)...")
enet_cv.fit(Xt_tr_dense, ytr)
log("Fit complete.")

log(f"*** ElasticNetCV 选出的最佳 Alpha: {enet_cv.alpha_:.4f} ***")
log(f"*** ElasticNetCV 选出的最佳 L1_Ratio: {enet_cv.l1_ratio_:.2f} ***")

# 5. 重新准备验证集 / 测试集的特征 (并转成 dense)
Xt_va_dense = _to_dense(preprocessor.transform(X_valid))
Xt_te_dense = _to_dense(preprocessor.transform(X_te))

# 6. 训练/验证集预测（log1p 尺度）
yhat_tr_log = enet_cv.predict(Xt_tr_dense)
yhat_va_log = enet_cv.predict(Xt_va_dense)

# 7. 计算 Log-scale 评估指标
r2_in  = r2_score(ytr, yhat_tr_log)
r2_out = r2_score(yva, yhat_va_log)
mse_in_log  = mean_squared_error(ytr, yhat_tr_log)
mse_out_log = mean_squared_error(yva, yhat_va_log)
rmse_in_log  = np.sqrt(mse_in_log)
rmse_out_log = np.sqrt(mse_out_log)


# vvvv [新增] 原始量纲评估指标计算 vvvv
# 7a. 转换回原始价格尺度
yhat_tr_orig = np.maximum(np.expm1(yhat_tr_log), 0.0)
yhat_va_orig = np.maximum(np.expm1(yhat_va_log), 0.0)

# 7b. 计算原始量纲的 MAE/MSE/RMSE
mae_in_orig  = mean_absolute_error(y_train, yhat_tr_orig)
mae_out_orig = mean_absolute_error(y_valid, yhat_va_orig)
mse_in_orig  = mean_squared_error(y_train, yhat_tr_orig)
mse_out_orig = mean_squared_error(y_valid, yhat_va_orig)
rmse_in_orig = np.sqrt(mse_in_orig)
rmse_out_orig = np.sqrt(mse_out_orig)
# ^^^^ [新增] 原始量纲评估指标计算 ^^^^


# 8. 打印指标 (!! 按照您的最终格式 !!)
print("\n" + "="*60)
print(f"--- ElasticNetCV (Alpha={enet_cv.alpha_:.4f}, L1r={enet_cv.l1_ratio_:.2f}) Metrics [log-scale R²] ---")
print("="*60)
print(f"In-sample R²:         {r2_in:.4f}")
print(f"Out-of-sample R²:     {r2_out:.4f}")
print("------------------------------------------------------------")
print(f"In-sample MSE (Log):  {mse_in_log:.6f}")
print(f"Out-of-sample MSE (Log): {mse_out_log:.6f}")
print(f"In-sample RMSE (Log): {rmse_in_log:.6f}")
print(f"Out-of-sample RMSE (Log): {rmse_out_log:.6f}")
print("="*60)

# vvvv [新增] 打印原始量纲的误差 vvvv
print("--- 原始量纲 (Original Scale) 误差 ---")
print(f"In-sample MAE:        {mae_in_orig:,.2f}")
print(f"Out-of-sample MAE:    {mae_out_orig:,.2f}")
print("-" * 60)
print(f"In-sample MSE:        {mse_in_orig:,.2f}")
print(f"Out-of-sample MSE:    {mse_out_orig:,.2f}")
print("-" * 60)
print(f"In-sample RMSE:       {rmse_in_orig:,.2f}")
print(f"Out-of-sample RMSE:   {rmse_out_orig:,.2f}")
print("="*60)
# ^^^^ [新增] 原始量纲评估指标计算 ^^^^


# 9. 预测测试集 (log1p 尺度) 并还原到原始价格尺度
y_pred_log = enet_cv.predict(Xt_te_dense) # <-- 使用 Xt_te_dense
y_pred     = np.expm1(y_pred_log)         # inverse of log1p
y_pred     = np.maximum(y_pred, 0.)        # clip 防止负值

# 10. 组提交 DataFrame (修正 ID/TARGET 逻辑)
ID_COL_UPPER = str(ID_COL).upper() 
ID_COL_LOWER = str(ID_COL).lower() 
TARGET_LOWER = str(TARGET).lower()

submission = pd.DataFrame({
    ID_COL_LOWER: df_test[ID_COL_UPPER], 
    TARGET_LOWER: y_pred
})

# 11. 保存提交文件
out = Path("reports/submission_ElasticNet_price.csv")
out.parent.mkdir(exist_ok=True)
submission.to_csv(out, index=False, encoding="utf-8-sig")

print("\n" + "="*50)
print(f"[SUBMIT] saved -> {out} | rows: {len(submission)}")
print("预测 {TARGET} 描述 (最终检查):")
print(submission[TARGET_LOWER].describe().apply('{:,.2f}'.format))
print("="*50)

# 12. 清理大矩阵，防止内存爆炸
del Xt_tr_dense, Xt_va_dense, Xt_te_dense
gc.collect()
log("已清理 dense 矩阵内存。")

[11:23:26] 模型 [I]: ElasticNetCV (自动调参 + 已加速) 开始...
[Check] ytr mean is 14.26 (log1p scale, OK)
[11:23:26] 将为 ElasticNet 测试 alpha: [0.0001 0.001  0.01   0.1   ]
[11:23:26] 将为 ElasticNet 测试 l1_ratio: [0.1, 0.5, 0.9]
[11:23:26] Fitting ElasticNetCV (cv=3, tol=0.001)...


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 103.01941871833515, tolerance: 38.17795415635479
  model = cd_fast.enet_coordinate_descent_gram(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 90.65126792408046, tolerance: 38.3028501163156
  model = cd_fast.enet_coordinate_descent_gram(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:664: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 94.68168639399846, tolerance: 38.36379026514998
  model = cd_fast.enet_coordinate_descent_gram(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:678

[11:25:18] Fit complete.
[11:25:18] *** ElasticNetCV 选出的最佳 Alpha: 0.0001 ***
[11:25:18] *** ElasticNetCV 选出的最佳 L1_Ratio: 0.10 ***

--- ElasticNetCV (Alpha=0.0001, L1r=0.10) Metrics [log-scale R²] ---
In-sample R²:         0.6509
Out-of-sample R²:     0.6408
------------------------------------------------------------
In-sample MSE (Log):  0.241268
Out-of-sample MSE (Log): 0.244434
In-sample RMSE (Log): 0.491191
Out-of-sample RMSE (Log): 0.494403
--- 原始量纲 (Original Scale) 误差 ---
In-sample MAE:        871,858.38
Out-of-sample MAE:    868,663.78
------------------------------------------------------------
In-sample MSE:        4,955,254,093,562.27
Out-of-sample MSE:    3,864,766,222,462.68
------------------------------------------------------------
In-sample RMSE:       2,226,040.00
Out-of-sample RMSE:   1,965,900.87

[SUBMIT] saved -> reports/submission_ElasticNet_price.csv | rows: 34017
预测 {TARGET} 描述 (最终检查):
count        34,017.00
mean      2,264,576.76
std       1,763,587.98
min     

In [51]:
# ===================================================================
# J.2. [最终] 合并 Price (OLS) 和 Rent (ElasticNet) 预测
# ===================================================================
log("开始合并 Price(OLS) 和 Rent(ElasticNet) 预测...")

# 1. 定义你的三个关键文件
TEMPLATE_FILE = "submission_template_Class25Q2.csv" # Kaggle 官方总模板
# !!! 确保你 rent 数据的 ElasticNet 预测文件名正确 !!!
RENT_PREDS_FILE = "reports/ElasticNet_rent_test.csv"     # 假设的 rent 数据 ElasticNet 预测文件名
PRICE_PREDS_FILE = "reports/submission_OLS_price.csv"  # 你的 price 数据 OLS 预测

# 2. 加载所有文件
try:
    df_template = pd.read_csv(TEMPLATE_FILE)
    df_rent_preds = pd.read_csv(RENT_PREDS_FILE)
    df_price_preds = pd.read_csv(PRICE_PREDS_FILE)
except FileNotFoundError as e:
    log(f"错误: 找不到文件 {e.filename}。")
    log("请确保 'ElasticNet_rent_test.csv' 和 'submission_OLS_price.csv' 都已存在。")
    raise

log(f"加载模板: {df_template.shape}")
log(f"加载 Rent 预测: {df_rent_preds.shape}")
log(f"加载 Price 预测: {df_price_preds.shape}")

# 3. (关键) 以总模板为基础，合并两个预测

# 3.1. 将两个预测文件转换为 "ID -> Price" 的字典
map_rent_price = df_rent_preds.set_index(ID_COL)[TARGET].to_dict()
map_price_price = df_price_preds.set_index(ID_COL)[TARGET].to_dict()

# 3.2. 更新：将 price 预测合并到 rent 字典中
map_rent_price.update(map_price_price)
all_preds_map = map_rent_price

# 4. 使用总模板的 ID 顺序，从我们的总字典中查找价格
final_submission_df = df_template[[ID_COL]].copy()
final_submission_df[TARGET] = final_submission_df[ID_COL].map(all_preds_map)

# 5. 检查并填充
missing_count = final_submission_df[TARGET].isna().sum()
if missing_count > 0:
    log(f"警告: 最终合并后仍有 {missing_count} 个 ID 缺少预测！")
    log("...正在用 0 填充...")
    final_submission_df[TARGET] = final_submission_df[TARGET].fillna(0)
else:
    log("合并成功！所有模板中的 ID 都找到了对应的预测。")

# 6. 导出最终文件
out = Path("reports/submission_FINAL_OLSprice_ENrent.csv"); out.parent.mkdir(exist_ok=True)
final_submission_df.to_csv(out, index=False, encoding="utf-8-sig")

print("\n" + "="*30)
print(f"saved -> {out} | rows: {len(final_submission_df)}")
print("最终预测 Price 描述 (已合并):")
print(final_submission_df[TARGET].describe())
print("="*30)

[15:04:59] 开始合并 Price(OLS) 和 Rent(ElasticNet) 预测...
[15:04:59] 加载模板: (43790, 2)
[15:04:59] 加载 Rent 预测: (9773, 2)
[15:04:59] 加载 Price 预测: (34017, 2)
[15:04:59] 合并成功！所有模板中的 ID 都找到了对应的预测。

saved -> reports/submission_FINAL_OLSprice_ENrent.csv | rows: 43790
最终预测 Price 描述 (已合并):
count    4.379000e+04
mean     1.935620e+06
std      1.906819e+06
min      6.693018e+04
25%      6.955883e+05
50%      1.242868e+06
75%      2.539665e+06
max      2.450998e+07
Name: Price, dtype: float64


In [40]:
# ===================================================================
# H. (代码生成) RENT 模型评估汇总表 (最终修复：强制显示 MAE 格式)
# ===================================================================
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

log("开始生成 RENT 模型评估汇总表 (MAE 强制显示)...")

# --- Helper Function for Formatting ---
def format_to_string(val, is_r2=False):
    if pd.isna(val):
        return 'N/A'
    elif is_r2:
        # R^2 格式 (4位小数)
        return f"{val:.4f}"
    else:
        # MAE 原始量纲格式 (带逗号和两位小数)
        return f"{val:,.2f}"

# 自定义函数：将带逗号的字符串 MAE 转换为浮点数（用于内部存储）
def parse_mae(s):
    if isinstance(s, str):
        return float(s.replace(',', ''))
    return s

# --- 1. 数据提取与格式化 ---
raw_data = {
    'OLS': { 
        'R2_in': 0.6906, 'R2_out': 0.6990, 'R2_cv': 0.6897,
        'MAE_in': parse_mae("188,736.56"),    
        'MAE_out': parse_mae("184,938.26")   
    },
    'RidgeCV': {
        'R2_in': 0.6906, 'R2_out': 0.6990, 'R2_cv': np.nan, 
        'MAE_in': parse_mae("188,716.28"),
        'MAE_out': parse_mae("184,925.88")
    },
    'LassoCV': {
        'R2_in': 0.6906, 'R2_out': 0.6990, 'R2_cv': np.nan, 
        'MAE_in': parse_mae("188,765.43"),
        'MAE_out': parse_mae("184,972.00")
    },
    'ElasticNetCV': {
        'R2_in': 0.6906, 'R2_out': 0.6990, 'R2_cv': np.nan, 
        'MAE_in': parse_mae("188,722.04"),
        'MAE_out': parse_mae("184,929.71")
    }
}

# --- 2. 构造最终的字典，使用字符串值 ---
final_data_dict = {}
model_names = ['OLS', 'RidgeCV', 'LassoCV', 'ElasticNetCV']

for model in model_names:
    d = raw_data[model]
    final_data_dict[model] = {
        'In-sample R²':             format_to_string(d['R2_in'], is_r2=True),
        'Out-of-sample R²':         format_to_string(d['R2_out'], is_r2=True),
        'Cross-validation R²':      format_to_string(d['R2_cv'], is_r2=True),
        'In-sample MAE (Original)': format_to_string(d['MAE_in']),
        'Out-of-sample MAE (Original)': format_to_string(d['MAE_out']),
    }

# --- 3. 创建 DataFrame ---
index_order = [
    'In-sample R²', 
    'Out-of-sample R²', 
    'Cross-validation R²',
    'In-sample MAE (Original)', 
    'Out-of-sample MAE (Original)'
]
df_rent_final = pd.DataFrame(final_data_dict).reindex(index_order)

# --- 4. 最终显示 ---
df_rent_final.columns = ['OLS', 'RidgeCV (α=1.0)', 'LassoCV (α=0.0001)', 'ENetCV (α=0.0001, L1r=0.10)']

log("生成表格完成。")

display(Markdown("### 📊 Rent 数据集模型评估汇总 (R² 为 Log Scale, MAE 为原始量纲)"))
display(df_rent_final)

[11:35:59] 开始生成 RENT 模型评估汇总表 (MAE 强制显示)...
[11:35:59] 生成表格完成。


### 📊 Rent 数据集模型评估汇总 (R² 为 Log Scale, MAE 为原始量纲)

,OLS,RidgeCV (α=1.0),LassoCV (α=0.0001),"ENetCV (α=0.0001, L1r=0.10)"
In-sample R²,0.6906,0.6906,0.6906,0.6906
Out-of-sample R²,0.6990,0.6990,0.6990,0.6990
Cross-validation R²,0.6897,N/A,N/A,N/A
In-sample MAE (Original),"188,736.56","188,716.28","188,765.43","188,722.04"
Out-of-sample MAE (Original),"184,938.26","184,925.88","184,972.00","184,929.71"


In [28]:
# ===================================================================
# H. (代码生成) PRICE 模型评估汇总表 (最终版：填入MAE数据)
# ===================================================================
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

log("开始生成 PRICE 模型评估汇总表 (填入MAE数据)...")

# ===================================================================
# 1. 精确数据提取 (来自您的日志)
#    - R²: Log Scale
#    - MAE: 原始量纲
# ===================================================================

price_data_final = {
    'OLS': { 
        'In-sample R²':         0.6510,
        'Out-of-sample R²':     0.6408,
        'Cross-validation R²':  -2.593234e+19, # OLS CV 爆炸
        'In-sample MAE (Orig)': 872135.55,    # <--- 填入数据
        'Out-of-sample MAE (Orig)': 868807.39   # <--- 填入数据
    },
    'RidgeCV': { # (Alpha=1.0)
        'In-sample R²':         0.6510,
        'Out-of-sample R²':     0.6408,
        'Cross-validation R²':  np.nan, 
        'In-sample MAE (Orig)': 872109.05,
        'Out-of-sample MAE (Orig)': 868726.10
    },
    'LassoCV': { # (Alpha=0.0001)
        'In-sample R²':         0.6507,
        'Out-of-sample R²':     0.6406,
        'Cross-validation R²':  np.nan, 
        'In-sample MAE (Orig)': 871907.59,
        'Out-of-sample MAE (Orig)': 868993.04
    },
    'ElasticNetCV': { # (Alpha=0.0001, L1r=0.10)
        'In-sample R²':         0.6509,
        'Out-of-sample R²':     0.6408,
        'Cross-validation R²':  np.nan, 
        'In-sample MAE (Orig)': 871858.38,
        'Out-of-sample MAE (Orig)': 868663.78
    }
}
# ===================================================================

# --- 2. 创建 DataFrame ---
index_order = [
    'In-sample R²', 
    'Out-of-sample R²', 
    'Cross-validation R²',
    'In-sample MAE (Orig)', 
    'Out-of-sample MAE (Orig)'
]
df_price_final = pd.DataFrame(price_data_final).reindex(index_order)

# --- 3. 定义格式化函数 ---
def format_df_final(df):
    df_formatted = df.copy().astype(object) 
    for col in df.columns:
        for idx in df.index:
            val = df.loc[idx, col]
            if pd.isna(val):
                df_formatted.loc[idx, col] = 'N/A'
            elif 'R²' in idx:
                # 特殊处理爆炸的 CV R²
                if idx == 'Cross-validation R²' and val < -1:
                   df_formatted.loc[idx, col] = '**Exploded!**' 
                else:
                   df_formatted.loc[idx, col] = f"{val:.4f}"
            elif 'MAE' in idx:
                # 原始量纲 MAE：保留两位小数和逗号
                df_formatted.loc[idx, col] = f"{val:,.2f}"
    return df_formatted

# --- 4. 格式化并显示 ---
df_price_formatted_final = format_df_final(df_price_final)

# 添加模型参数到列名
df_price_formatted_final.columns = ['OLS', 'RidgeCV (α=1.0)', 'LassoCV (α=0.0001)', 'ENetCV (α=0.0001, L1r=0.10)']

log("生成表格完成。")

display(Markdown("### 📊 Price 数据集模型评估汇总 (R² 为 Log Scale, MAE 为原始量纲)"))
display(df_price_formatted_final)

[11:29:27] 开始生成 PRICE 模型评估汇总表 (填入MAE数据)...
[11:29:27] 生成表格完成。


### 📊 Price 数据集模型评估汇总 (R² 为 Log Scale, MAE 为原始量纲)

,OLS,RidgeCV (α=1.0),LassoCV (α=0.0001),"ENetCV (α=0.0001, L1r=0.10)"
In-sample R²,0.6510,0.6510,0.6507,0.6509
Out-of-sample R²,0.6408,0.6408,0.6406,0.6408
Cross-validation R²,**Exploded!**,N/A,N/A,N/A
In-sample MAE (Orig),"872,135.55","872,109.05","871,907.59","871,858.38"
Out-of-sample MAE (Orig),"868,807.39","868,726.10","868,993.04","868,663.78"


In [36]:
# ===================================================================
# I. (代码生成) Kaggle 分数汇总表
# ===================================================================
import pandas as pd
from IPython.display import display, Markdown

log("开始生成 Kaggle 分数汇总表...")

# ===================================================================
# 1. [!!! 用户输入区 !!!] 
# 
# 请将下面的分数替换为您实际的 Kaggle Score
# ===================================================================

kaggle_scores = {
    'OLS':        54.73,
    'Lasso':      54.26,
    'Ridge':      54.68,
    'ElasticNet': 54.47,
}
# ===================================================================

# --- 2. 创建 DataFrame ---
# (我们转置一下，让模型名作为索引)
df_kaggle = pd.DataFrame.from_dict(kaggle_scores, orient='index', columns=['Kaggle Score'])

# --- 3. 格式化分数 (保留两位小数) ---
df_kaggle_formatted = df_kaggle.copy()
df_kaggle_formatted['Kaggle Score'] = df_kaggle_formatted['Kaggle Score'].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "N/A")

# --- 4. 修改索引名 ---
df_kaggle_formatted.index.name = '模型'

log("生成表格完成。")

# --- 5. 最终输出 ---
display(Markdown("### 🏆 Kaggle 分数汇总"))
display(df_kaggle_formatted)

# --- 6. (可选) 输出 Markdown 源码 ---
# print("\nMarkdown 源码:\n")
# print(df_kaggle_formatted.to_markdown())

[10:12:34] 开始生成 Kaggle 分数汇总表...
[10:12:34] 生成表格完成。


### 🏆 Kaggle 分数汇总

,Kaggle Score
模型,
OLS,54.73
Lasso,54.26
Ridge,54.68
ElasticNet,54.47


# 